In [7]:
!pip install requests beautifulsoup4 selenium tqdm

In [59]:
!D:\conda\python.exe -m pip install --upgrade pip

In [28]:
!pip install selenium webdriver-manager

In [2]:
import re
import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, unquote

# Crawling from Cookpad website

In [3]:
city_location_map = {
    # Kermanshah Location
    "کرمانشاه":      {"city": "کرمانشاه",      "latitude": 34.3142, "longitude": 47.0650},
    "اسلام‌آباد غرب": {"city": "اسلام‌آباد غرب", "latitude": 34.1111, "longitude": 46.5278},
    "جوانرود":      {"city": "جوانرود",      "latitude": 34.8061, "longitude": 46.4889},
    "کنگاور":       {"city": "کنگاور",       "latitude": 34.5042, "longitude": 47.9650},
    "سرپل ذهاب":     {"city": "سرپل ذهاب",     "latitude": 34.4572, "longitude": 45.8611},
    "سنقر":         {"city": "سنقر",         "latitude": 34.7833, "longitude": 47.6000},
    "هرسین":        {"city": "هرسین",        "latitude": 34.2711, "longitude": 47.5861},
    "صحنه":         {"city": "صحنه",         "latitude": 34.4811, "longitude": 47.6833},
    "پاوه":         {"city": "پاوه",         "latitude": 35.0431, "longitude": 46.3561},
    "روانسر":       {"city": "روانسر",       "latitude": 34.7167, "longitude": 46.6500},
    "گیلانغرب":     {"city": "گیلانغرب",     "latitude": 34.1397, "longitude": 45.9200},
    "قصر شیرین":     {"city": "قصر شیرین",     "latitude": 34.5150, "longitude": 45.5772},
    "تازه‌آباد":     {"city": "تازه‌آباد",     "latitude": 34.7442, "longitude": 46.1511},
    "کرند غرب":      {"city": "کرند غرب",      "latitude": 34.2833, "longitude": 46.2333},
    "سورانه":        {"city": "سورانه",        "latitude": 34.4425, "longitude": 45.8431},
    "باینگان":       {"city": "باینگان",       "latitude": 34.9667, "longitude": 46.2833},
    "ثلاث باباجانی": {"city": "ثلاث باباجانی", "latitude": 34.7358, "longitude": 46.1494},

    # Ilam Location
    "ایلام":         {"city": "ایلام",         "latitude": 33.6374, "longitude": 46.4227},
    "دهلران":       {"city": "دهلران",       "latitude": 32.7450, "longitude": 47.2644},
    "آبدانان":      {"city": "آبدانان",      "latitude": 33.1561, "longitude": 47.4328},
    "دره‌شهر":      {"city": "دره‌شهر",      "latitude": 33.1383, "longitude": 47.3786},
    "ایوان":        {"city": "ایوان",        "latitude": 33.8275, "longitude": 46.3014},
    "مهران":        {"city": "مهران",        "latitude": 33.1336, "longitude": 46.1839},
    "لومار":        {"city": "لومار",        "latitude": 33.5675, "longitude": 46.8142},
    "چرداول":       {"city": "چرداول",       "latitude": 33.8333, "longitude": 46.8333},
    "سرابله":       {"city": "سرابله",       "latitude": 33.4639, "longitude": 46.5664},
    "موسیان":       {"city": "موسیان",       "latitude": 33.4833, "longitude": 46.8000},

    # Lorestan Location
    "خرم‌آباد":     {"city": "خرم‌آباد",     "latitude": 33.4878, "longitude": 48.3558},
    "بروجرد":       {"city": "بروجرد",       "latitude": 33.8972, "longitude": 48.7518},
    "دورود":        {"city": "دورود",        "latitude": 33.4950, "longitude": 49.0236},
    "کوهدشت":       {"city": "کوهدشت",       "latitude": 33.5295, "longitude": 47.6136},
    "ازنا":         {"city": "ازنا",         "latitude": 33.4558, "longitude": 49.4356},
    "الشتر":        {"city": "الشتر",        "latitude": 33.8442, "longitude": 48.2689},
    "پلدختر":       {"city": "پلدختر",       "latitude": 33.1372, "longitude": 47.7133},
    "الیگودرز":     {"city": "الیگودرز",     "latitude": 33.3971, "longitude": 49.7010},
    "نورآباد":      {"city": "نورآباد",      "latitude": 34.0733, "longitude": 47.9725},
    "چغلوندی":      {"city": "چغلوندی",      "latitude": 33.6500, "longitude": 48.7000},
    "زاغه":         {"city": "زاغه",         "latitude": 33.5000, "longitude": 48.7167},
    "سپیددشت":      {"city": "سپیددشت",      "latitude": 33.1833, "longitude": 47.7667},

    # Khuzestan Location
    "اهواز":        {"city": "اهواز",        "latitude": 31.3183, "longitude": 48.6706},
    "آبادان":       {"city": "آبادان",       "latitude": 30.3392, "longitude": 48.3043},
    "خرمشهر":       {"city": "خرمشهر",       "latitude": 30.4394, "longitude": 48.1817},
    "دزفول":        {"city": "دزفول",        "latitude": 32.3811, "longitude": 48.4058},
    "شوشتر":        {"city": "شوشتر",        "latitude": 32.0450, "longitude": 48.8594},
    "مسجدسلیمان":   {"city": "مسجدسلیمان",   "latitude": 31.9543, "longitude": 49.2853},
    "رامهرمز":      {"city": "رامهرمز",      "latitude": 31.2782, "longitude": 49.5891},
    "بهبهان":       {"city": "بهبهان",       "latitude": 30.5953, "longitude": 50.2431},
    "اندیمشک":      {"city": "اندیمشک",      "latitude": 32.4600, "longitude": 48.3594},
    "شوش":          {"city": "شوش",          "latitude": 32.1942, "longitude": 48.2436},
    "هفتکل":        {"city": "هفتکل",        "latitude": 31.4453, "longitude": 49.5253},
    "لالی":         {"city": "لالی",         "latitude": 32.3289, "longitude": 49.0936},
    "باغ‌ملک":      {"city": "باغ‌ملک",      "latitude": 31.5231, "longitude": 49.8861},
    "هویزه":        {"city": "هویزه",        "latitude": 31.4653, "longitude": 48.0789},
    "حمیدیه":       {"city": "حمیدیه",       "latitude": 31.4833, "longitude": 48.4333},

    # Hamadan Location
    "همدان":        {"city": "همدان",        "latitude": 34.7986, "longitude": 48.5146},
    "ملایر":        {"city": "ملایر",        "latitude": 34.3143, "longitude": 48.8210},
    "نهاوند":       {"city": "نهاوند",       "latitude": 34.2005, "longitude": 48.3758},
    "تویسرکان":     {"city": "تویسرکان",     "latitude": 34.5497, "longitude": 48.4497},
    "رزن":          {"city": "رزن",          "latitude": 35.4108, "longitude": 49.0127},
    "کبودرآهنگ":    {"city": "کبودرآهنگ",    "latitude": 35.2203, "longitude": 48.1856},
    "بهار":         {"city": "بهار",         "latitude": 34.8997, "longitude": 48.4336},
    "فامنین":       {"city": "فامنین",       "latitude": 35.0121, "longitude": 48.9876},
    "اسدآباد":      {"city": "اسدآباد",      "latitude": 34.7981, "longitude": 47.9890},
    "قهاوند":       {"city": "قهاوند",       "latitude": 34.7833, "longitude": 48.5833},
    "لالجین":       {"city": "لالجین",       "latitude": 35.0167, "longitude": 48.5167},

    # Markazi Location
    "اراک":         {"city": "اراک",         "latitude": 34.0917, "longitude": 49.6892},
    "ساوه":         {"city": "ساوه",         "latitude": 35.0214, "longitude": 50.3568},
    "خمین":         {"city": "خمین",         "latitude": 33.6414, "longitude": 50.0780},
    "محلات":        {"city": "محلات",        "latitude": 33.9208, "longitude": 50.4271},
    "شازند":        {"city": "شازند",        "latitude": 33.9356, "longitude": 49.4211},
    "تفرش":         {"city": "تفرش",         "latitude": 34.6936, "longitude": 50.0161},
    "کمیجان":       {"city": "کمیجان",       "latitude": 34.7164, "longitude": 49.3870},
    "زرندیه":       {"city": "زرندیه",       "latitude": 35.2139, "longitude": 50.4969},
    "آشتیان":       {"city": "آشتیان",       "latitude": 34.5219, "longitude": 50.0119},
    "دلیجان":       {"city": "دلیجان",       "latitude": 33.9906, "longitude": 50.6839},
    "خنداب":        {"city": "خنداب",        "latitude": 34.3833, "longitude": 49.1833},
    "فراهان":       {"city": "فراهان",       "latitude": 34.5094, "longitude": 49.6919},
    "مامونیه":      {"city": "مامونیه",      "latitude": 35.3000, "longitude": 50.5000},
}

In [4]:
suffix_location_map = {
    # Kermanshah Province
    "کرمانشاهی": ("کرمانشاه", "کرمانشاه"),
    "اسلام‌آبادی": ("کرمانشاه", "اسلام‌آباد غرب"),
    "جوانرودی": ("کرمانشاه", "جوانرود"),
    "کنگاوری": ("کرمانشاه", "کنگاور"),
    "سرپل‌ذهابی": ("کرمانشاه", "سرپل ذهاب"),
    "سنقری": ("کرمانشاه", "سنقر"),
    "هرسینی": ("کرمانشاه", "هرسین"),
    "صحنه‌ای": ("کرمانشاه", "صحنه"),
    "پاوه‌ای": ("کرمانشاه", "پاوه"),
    "روانسری": ("کرمانشاه", "روانسر"),
    "گیلانغربی": ("کرمانشاه", "گیلانغرب"),
    "قصرشیرینی": ("کرمانشاه", "قصر شیرین"),
    "تازه‌آبادی": ("کرمانشاه", "تازه‌آباد"),
    "کرندی": ("کرمانشاه", "کرند غرب"),
    "سورانه‌ای": ("کرمانشاه", "سورانه"),
    "باینگانی": ("کرمانشاه", "باینگان"),
    "ثلاث‌باباجانی": ("کرمانشاه", "ثلاث باباجانی"),

    # Ilam Province
    "ایلامی": ("ایلام", "ایلام"),
    "دهلرانی": ("ایلام", "دهلران"),
    "آبدانانی": ("ایلام", "آبدانان"),
    "دره‌شهری": ("ایلام", "دره‌شهر"),
    "ایوانی": ("ایلام", "ایوان"),
    "مهرانی": ("ایلام", "مهران"),
    "لوماری": ("ایلام", "لومار"),
    "چرداولی": ("ایلام", "چرداول"),
    "سرابله‌ای": ("ایلام", "سرابله"),
    "موسیانی": ("ایلام", "موسیان"),

    # Lorestan Province
    "خرم‌آبادی": ("لرستان", "خرم‌آباد"),
    "بروجردی": ("لرستان", "بروجرد"),
    "دورودی": ("لرستان", "دورود"),
    "کوهدشتی": ("لرستان", "کوهدشت"),
    "ازنایی": ("لرستان", "ازنا"),
    "الشتری": ("لرستان", "الشتر"),
    "پلدختری": ("لرستان", "پلدختر"),
    "الیگودرزی": ("لرستان", "الیگودرز"),
    "نورآبادی": ("لرستان", "نورآباد"),
    "چغلوندی": ("لرستان", "چغلوندی"),
    "زاغه‌ای": ("لرستان", "زاغه"),
    "سپیددشتی": ("لرستان", "سپیددشت"),

    # Khuzestan Province
    "اهوازی": ("خوزستان", "اهواز"),
    "آبادانی": ("خوزستان", "آبادان"),
    "خرمشهری": ("خوزستان", "خرمشهر"),
    "دزفولی": ("خوزستان", "دزفول"),
    "شوشتری": ("خوزستان", "شوشتر"),
    "مسجدسلیمانی": ("خوزستان", "مسجدسلیمان"),
    "رامهرمزی": ("خوزستان", "رامهرمز"),
    "بهبهانی": ("خوزستان", "بهبهان"),
    "اندیمشکی": ("خوزستان", "اندیمشک"),
    "شوشی": ("خوزستان", "شوش"),
    "هفتکلی": ("خوزستان", "هفتکل"),
    "لالی": ("خوزستان", "لالی"),
    "باغ‌ملکی": ("خوزستان", "باغ‌ملک"),
    "هویزه‌ای": ("خوزستان", "هویزه"),
    "حمیدیه‌ای": ("خوزستان", "حمیدیه"),

    # Hamadan Province
    "همدانی": ("همدان", "همدان"),
    "ملایری": ("همدان", "ملایر"),
    "نهاوندی": ("همدان", "نهاوند"),
    "تویسرکانی": ("همدان", "تویسرکان"),
    "رزنی": ("همدان", "رزن"),
    "کبودرآهنگی": ("همدان", "کبودرآهنگ"),
    "بهاری": ("همدان", "بهار"),
    "فامنینی": ("همدان", "فامنین"),
    "اسدآبادی": ("همدان", "اسدآباد"),
    "قهاوندی": ("همدان", "قهاوند"),
    "لالجینی": ("همدان", "لالجین"),

    # Markazi Province
    "اراکی": ("مرکزی", "اراک"),
    "ساوه‌ای": ("مرکزی", "ساوه"),
    "خمینی": ("مرکزی", "خمین"),
    "محلاتی": ("مرکزی", "محلات"),
    "شازندی": ("مرکزی", "شازند"),
    "تفرشی": ("مرکزی", "تفرش"),
    "کمیجانی": ("مرکزی", "کمیجان"),
    "زرندیه‌ای": ("مرکزی", "زرندیه"),
    "آشتیانی": ("مرکزی", "آشتیان"),
    "دلیجانی": ("مرکزی", "دلیجان"),
    "خندابی": ("مرکزی", "خنداب"),
    "فراهانی": ("مرکزی", "فراهان"),
    "مامونیه‌ای": ("مرکزی", "مامونیه"),
}

In [5]:
def detect_meal_type(title, ingredients_text):
    title_lower = title.lower()
    ingredients_lower = ingredients_text.lower() if ingredients_text else ""
    
    if any(w in title_lower for w in ["آبگوشت", "جگر", "خورشت", "خورش", "کباب", "کوفته", "دلمه", "کتلت", "شامی"]):
        return ["غذای اصلی گوشتی"]
    
    if any(w in title_lower for w in ["پلو", "برنج", "برنجی", "چلو"]):
        return ["غذای اصلی برنجی"]
    
    if any(w in title_lower for w in ["آش", "سوپ", "حلیم", "اشکنه"]):
        return ["آش و سوپ"]
    
    if any(w in title_lower for w in ["خوراک", "بورانی", "کوکو", "املت"]):
        if "گوشت" in ingredients_lower:
            return ["غذای اصلی گوشتی"]
        return ["غذای اصلی گیاهی"]
    
    if any(w in title_lower for w in ["شیرینی", "حلوا", "کیک", "بستنی", "دسر", "شله", "مهلبیه"]):
        return ["دسر"]
    
    if any(w in title_lower for w in ["نان", "ساجی", "لواش", "تافتون"]):
        return ["نان و شیرینی"]
    
    if any(w in title_lower for w in ["سالاد", "ترشی", "ماست"]):
        return ["پیش غذا و مکمل"]
    

    if "گوشت" in ingredients_lower:
        return ["غذای اصلی گوشتی"]
    
    if "برنج" in ingredients_lower:
        return ["غذای اصلی برنجی"]
    
    if "شکر" in ingredients_lower or "عسل" in ingredients_lower:
        return ["دسر"]
    
    return ["-"]


In [6]:
def detect_occasion(title, meal_type, ingredients=None, tags=None, season=None, time_of_day=None, is_weekend=False):
    title_lower = title.lower()
    ingredients_lower = ingredients.lower() if ingredients else ""
    tags_lower = [tag.lower() for tag in tags] if tags else []

    MAIN_OCCASIONS = {
        "صبحانه": ["صبحانه", "ناشتا", "کله", "پاچه", "حلیم", "املت", "نیمرو", "پنیر"],
        "ناهار": ["ناهار", "چاشت", "ظهر", "پلو", "برنج", "چلو", "خورشت"],
        "شام": ["شام", "شبان", "کباب", "آبگوشت", "دیزی"],
        "عصرانه": ["عصرانه", "چای", "کیک", "شیرینی", "دسر"],
        "سحری": ["سحری", "رمضان"],
        "افطاری": ["افطاری", "افطار"]
    }

    primary_occasion = None

    # S-specific occasions first
    if any(kw in title_lower for kw in MAIN_OCCASIONS["سحری"]):
        return "سحری"
    if any(kw in title_lower for kw in MAIN_OCCASIONS["افطاری"]):
        return "افطاری"

    # Check main occasions
    for occasion, keywords in MAIN_OCCASIONS.items():
        if any(kw in title_lower for kw in keywords):
            primary_occasion = occasion
            break

    # Default fallback based on time_of_day or weekend
    if not primary_occasion:
        if time_of_day == "صبح":
            return "صبحانه"
        if time_of_day == "عصر":
            return "عصرانه"
        return "شام" if not is_weekend else "ناهار"

    return primary_occasion

In [7]:
WORD_TO_NUMBER = {
    "نیم": 0.5,
    "نصف": 0.5,
    "ربع": 0.25,
    "یک": 1,
    "دو": 2,
    "سه": 3,
    "چهار": 4,
    "پنج": 5,
    "شش": 6,
    "هفت": 7,
    "هشت": 8,
    "نه": 9,
    "ده": 10,
    "½": 0.5,
    "¾": 0.75,
    "1/2": 0.5,
    "1/4": 0.25,
    "1/3": 0.66,
}

UNIT_WORDS = [
    "پیمانه", "عدد", "قاشق غذاخوری", "قاشق چای‌خوری", "قاشق سوپخوری",
    "قاشق مرباخوری", "کاسه", "بشقاب", "لیوان", "فنجان",
    "گرم", "کیلو", "حبه", "ق غ","ق.غ", "ق.چ","ق چ", "ق م", "پ","ق غذاخوری","ق چای خوری","قاشق غ","قاشق چای خوری","پنس","استکان""قاشق غذا خوری"
]

DESC_PATTERNS = [
    "به مقدار لازم", "به میزان لازم", "به میزان کافی",
    "به اندازه لازم", "مقداری", "به مقدار دلخواه", "کمی","بمقدار لازم", "به‌ اندازه نياز","به میزان دلخواه","مقدار لازم"
]

RANGE_PATTERN = re.compile(r"(\d+)\s*(?:الی|تا)\s*\d+")
NORMALIZE_DIGITS = str.maketrans(
    '۰۱۲۳۴۵۶۷۸۹٫،',
    '0123456789.,'
)
SEPARATE_PATTERN = re.compile(r"(?<=\d)(?=\D)|(?<=\D)(?=\d)")
PARSE_PATTERN = re.compile(
    rf"(?P<amount>\d+(?:\.\d+)?)\s*(?P<unit>({'|'.join(map(re.escape, UNIT_WORDS))}))?\s+"
    r"(?P<name>[\u0600-\u06FFa-zA-Z0-9\s\-()]+)"
)

def parse_ingredient_line(line: str) -> dict:
   
    for desc in DESC_PATTERNS:
        if desc in line:
            name = re.sub(r'[.:()۰-۹\d]+' , '', line.replace(desc, '')).strip(' :-–،')
            return {"name": name, "amount": desc, "unit": None}

    # Normalize digits and punctuation & Simplify ranges & Separate digits from letters
    normalized = line.translate(NORMALIZE_DIGITS)
    normalized = RANGE_PATTERN.sub(r"\1", normalized)
    separated = SEPARATE_PATTERN.sub(' ', normalized)

    # Convert textual numbers
    words = separated.split()
    for i, w in enumerate(words):
        if w in WORD_TO_NUMBER:
            words[i] = str(WORD_TO_NUMBER[w])
    cleaned = ' '.join(words)

    match = PARSE_PATTERN.search(cleaned)
    if match:
        amount = float(match.group('amount'))
        unit = match.group('unit')
        name = match.group('name').strip()
        if name in UNIT_WORDS:
            return {"name": line.strip(), "amount": None, "unit": None}
        return {"name": name, "amount": amount, "unit": unit}

    return {"name": line.strip(), "amount": None, "unit": None}


In [8]:
def extract_city_from_url(path: str, title: str = "") -> list:

    # URL-based suffixes between hyphens
    url_suffixes = re.findall(r'-([\u0600-\u06FF()]+)', path)
    candidates = [s.strip('()') for s in url_suffixes]
    # Title-based parentheses
    candidates += re.findall(r'\((.*?)\)', title)
    # Title last word
    parts = title.strip().split()
    if parts:
        candidates.append(parts[-1])
    # Deduplicate preserving order
    seen = []
    for c in candidates:
        if c and c not in seen:
            seen.append(c)
    # Map to province/city
    results = []
    for suffix in seen:
        if suffix in suffix_location_map:
            prov, city = suffix_location_map[suffix]
            results.append({'province': prov, 'city': city})
    # Fallback default
    return results 

In [9]:
def split_instructions(instructions: str) -> list:
    """Split raw instruction string into sentences."""
    parts = re.split(r'[.!؟]\s*', instructions)
    return [s.strip() for s in parts if len(s.strip()) > 5]

In [10]:
def extract_cookpad_recipe(url: str, province_override: str = None) -> dict:
    resp = requests.get(url)
    soup = BeautifulSoup(resp.content, 'html.parser')

    script = soup.find(
        'script', type='application/ld+json',
        string=lambda s: 'Recipe' in s if s else False
    )
    if not script:
        return None
    data = json.loads(script.string)

    title = data.get('name', 'نامشخص')
    raw_path = unquote(urlparse(url).path)

    locs = extract_city_from_url(raw_path, title)
    if province_override:
        for loc in locs:
            loc['province'] = province_override

    enriched = []
    for loc in locs:
        prov = loc.get('province')
        city = loc.get('city')
        # assign default city if none
        if not city:
            if prov == 'خوزستان':
                city = 'اهواز'
            elif prov in ('مرکزی', 'Markazi'):  # ensure both Persian and English
                city = 'اراک'
            elif prov in ('لرستان'):  # ensure both Persian and English
                city = 'خرم‌آباد'
            else:
                city = prov
        coords = city_location_map.get(city, {})
        enriched.append({
            'province': prov,
            'city': city,
            'latitude': coords.get('latitude'),
            'longitude': coords.get('longitude')
        })

    # parse ingredients, instructions, etc.
    ingredients_raw = data.get('recipeIngredient', [])
    ingredients = [parse_ingredient_line(i) for i in ingredients_raw]
    ingredients_text = '\n'.join(ingredients_raw)

    instr_raw = data.get('recipeInstructions', [])
    if isinstance(instr_raw, list):
        instructions = [step['text'] if isinstance(step, dict) else step
                        for step in instr_raw]
    else:
        instructions = split_instructions(instr_raw)

    image = data.get('image', '')
    if isinstance(image, list):
        image = image[0]

    meal_type = detect_meal_type(title, ingredients_text)
    occasion = detect_occasion(title, meal_type)

    return {
        'title': title,
        'locations': enriched,
        'ingredients': ingredients,
        'instructions': instructions,
        'meal_type': meal_type,
        'occasion': occasion,
        'images': {
            'main': image,
            'step1': None,
            'step2': None
        }
    }

In [11]:
def process_recipes_by_province(province: str, urls: list) -> list:
    results = []
    for url in urls:
        try:
            recipe = extract_cookpad_recipe(url, province_override=province)
            if recipe:
                results.append(recipe)
        except Exception as e:
            #print(f" Error processing URL: {url}\n   → {e}")
            continue
    return results

In [13]:
def main():
    
    all_urls = {
        "کرمانشاه": [
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/22597359-%D8%A2%D8%A8%DA%AF%D9%88%D8%B4%D8%AA-%D8%A8%D8%A7%D8%BA%DB%8C-%D8%A8%D8%A7-%DA%AF%D9%88%D8%B4%D8%AA-%DA%AF%D9%88%D8%B3%D9%81%D9%86%D8%AF-%D9%88-%D8%B9%D9%86%D8%A7%D8%A8?ref=search&search_term=%D8%A2%D8%A8%DA%AF%D9%88%D8%B4%D8%AA+%D8%A8%D8%A7%D8%BA%DB%8C",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/24502377-%D8%A2%D8%B4-%D8%A8%D9%84%D8%BA%D9%88%D8%B1?ref=search&search_term=%D8%A2%D8%B4+%D8%A8%D9%84%D8%BA%D9%88%D8%B1",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/16957558-%D8%A2%D8%B4-%D8%AA%D8%B1%D8%AE%DB%8C%D9%86%D9%87?ref=search&search_term=%D8%A2%D8%B4+%D8%AA%D8%B1%D8%AE%DB%8C%D9%86%D9%87+%D8%AF%D9%88%D8%BA",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/24228216-%D8%A2%D8%B4-%D8%AF%D9%88%D8%BA-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87?ref=search&search_term=%D8%A2%D8%B4+%D8%AF%D9%88%D8%BA+%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/24521539-%D8%A2%D8%B4-%D9%85%D8%A7%D8%B3%D8%AA-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/14173159-%D8%A2%D8%B4-%D9%85%D8%AD%D9%84%DB%8C-%D8%B3%D9%88%D8%B1%D8%A7%D9%86%D9%87-%DB%8C%D8%A7%D8%B3%D8%A7%D9%82%D9%87-%D8%B3%D8%B1%D8%AE?ref=search&search_term=%D8%A2%D8%B4+%D8%B3%D9%88%D8%B1%D8%A7%D9%86%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/17151326-%D8%A2%D8%B4-%D8%B9%D8%A8%D8%A7%D8%B3%D8%B9%D9%84%DB%8C-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87?ref=search&search_term=%D8%A2%D8%B4+%D8%B9%D8%A8%D8%A7%D8%B3%D8%B9%D9%84%DB%8C",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/13929977-%D8%AC%D8%B2-%D8%A8%D8%B2-%D8%AC%D8%BA%D9%88%D8%B1-%D8%A8%D8%BA%D9%88%D8%B1?ref=search&search_term=%D8%AC%DA%AF%D8%B1+%D9%88+%D8%A8%D8%B2",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/17250542-%D8%AE%D9%88%D8%B1%D8%B4%D8%AA-%D8%AE%D9%84%D8%A7%D9%84-%D8%A8%D8%A7%D8%AF%D8%A7%D9%85-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87?ref=search&search_term=%D8%AE%D9%88%D8%B1%D8%B4+%D8%AE%D9%84%D8%A7%D9%84+%D8%A8%D8%A7%D8%AF%D8%A7%D9%85",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/24094212-%D8%AE%D9%88%D8%B1%D8%B4-%DA%A9%D9%86%DA%AF%D8%B1?ref=search&search_term=%D8%AE%D9%88%D8%B1%D8%B4+%DA%A9%D9%86%DA%AF%D8%B1",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/10741049-%D8%AF%D8%B3%D8%AA-%D9%BE%DB%8C%DA%86-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C?ref=search&search_term=%D8%AF%D8%B3%D8%AA+%D9%BE%DB%8C%DA%86+%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/1717637-%D8%AF%D9%86%D8%AF%D9%87-%DA%A9%D8%A8%D8%A7%D8%A8-%D9%85%D8%AE%D8%B5%D9%88%D8%B5-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87?ref=search&search_term=%D8%AF%D9%86%D8%AF%D9%87+%DA%A9%D8%A8%D8%A7%D8%A8+%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/14318751-%D8%B3%DB%8C%D8%A8%D9%BE%D9%84%D9%88-%D8%A8%D8%A7-%D9%82%D8%A7%D8%B1%DA%86?ref=search&search_term=%D8%B3%DB%8C%D8%A8%E2%80%8C%D9%BE%D9%84%D9%88",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/24626571-%D8%B4%DB%8C%D8%B1-%D8%A8%D8%B1%D9%86%D8%AC?ref=search&search_term=%D8%B4%DB%8C%D8%B1+%D8%A8%D8%B1%D9%86%D8%AC",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/17191302-%DA%A9%D9%88%D9%81%D8%AA%D9%87-%D8%B1%DB%8C%D8%B2%D9%87-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87?ref=search&search_term=%DA%A9%D9%88%D9%81%D8%AA%D9%87+%D8%B1%DB%8C%D8%B2%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/24651077-%DA%A9%D9%88%D9%81%D8%AA%D9%87-%D9%86%D8%AE%D9%88%D8%AF?ref=search&search_term=%DA%A9%D9%88%D9%81%D8%AA%D9%87+%D9%86%D8%AE%D9%88%D8%AF",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/16359653-%D9%87%D9%84%D9%88-%DA%A9%D8%A8%D8%A7%D8%A8?ref=search&search_term=%D9%87%D9%84%D9%88+%DA%A9%D8%A8%D8%A7%D8%A8",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/13506269-%D9%88%D9%86%D9%88%D8%B4%DA%A9-%D9%BE%D9%84%D9%88?ref=search&search_term=%D9%88%D9%86%D9%88%D8%B4%DA%A9+%D9%BE%D9%84%D9%88",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/17229998-%D8%A8%DA%98%DB%8C-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/24158014-%D8%B3%DB%8C%D8%A8-%D9%BE%D9%84%D9%88%DB%8C-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/22556927-%D9%86%D8%A7%D9%86-%D8%B1%D9%88%D8%BA%D9%86%DB%8C-%D8%A7%D8%B5%D9%84-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/17314025-%DA%A9%D8%A7%DA%86%DB%8C-%D9%82%DB%8C%D9%85%D8%A7%D9%82-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C-%D8%A8%D8%A7-%D8%A2%D8%B1%D8%AF-%DA%AF%D9%86%D8%AF%D9%85?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/17270015-%D8%AE%D8%B1%D9%85%D8%A7-%D9%88%D8%B1%D9%88%DB%8C%D9%86%D8%AE%D8%B1%D9%85%D8%A7-%D8%A8%D8%A7-%D8%B1%D9%88%D8%BA%D9%86-%D8%AD%DB%8C%D9%88%D8%A7%D9%86%DB%8C-%D8%A7%D8%B5%D9%84-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/15441664-%D8%AF%D9%86%D8%AF%D9%87-%D9%BE%D9%84%D9%88-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/15083643-%D9%86%D8%A7%D9%86-%D8%A8%D8%B1%D9%86%D8%AC%DB%8C-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/14809886-%D9%86%D8%A7%D9%86-%D8%AE%D8%B1%D9%85%D8%A7%DB%8C%DB%8C-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/12986953-%DA%A9%D8%A7%DA%A9-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C?ref=search&search_term=%DA%A9%D8%A7%DA%A9+%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/15057735-%DA%A9%D9%88%DA%A9%D9%88-%D8%B4%DB%8C%D8%B1-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/15026726-%DA%A9%D9%88%D9%81%D8%AA%D9%87-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/14839799-%D8%AD%D9%84%D9%88%D8%A7%DB%8C-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/14376995-%DA%A9%D8%B4%DA%A9-%D8%A8%D8%A7%D8%AF%D9%85%D8%AC%D9%88%D9%86-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/14179407-%DA%AF%D8%B1%D8%AF%D9%87-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C-%D9%85%D8%A7%D8%AF%D8%B1%D8%B4%D9%88%D9%87%D8%B1%D9%85-%D8%AF%D8%B1%D8%B3%D8%AA-%DA%A9%D8%B1%D8%AF%D9%87?ref=search&search_term=%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/16235778-%D8%B4%D8%A7%D9%85%DB%8C-%DA%A9%D8%A8%D8%A7%D8%A8-%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C-%D8%A8%D8%A7-%D8%B3%D8%A7%D9%84%D8%A7%D8%AF-%D8%B4%DB%8C%D8%B1%D8%A7%D8%B2%DB%8C?ref=search&search_term=%D8%B4%D8%A7%D9%85%DB%8C+%DA%A9%D8%A8%D8%A7%D8%A8+%DA%A9%D8%B1%D9%85%D8%A7%D9%86%D8%B4%D8%A7%D9%87%DB%8C"
],
        "ایلام": [
  "https://cookpad.com/ir/دستور%20غذا/17056007-آبگوشت-بامیه-ایلامی-در-قابلمه-مسی?ref=search&search_term=آبگوشت+بامیه",
  "https://cookpad.com/ir/دستور%20غذا/24519734-آش-شله-امیری-ایلام?ref=search&search_term=شله+امیری",
  "https://cookpad.com/ir/دستور%20غذا/12987337-دنده-پلو-ایلام?ref=search&search_term=دنده+پلو",
  "https://cookpad.com/ir/دستور%20غذا/12977677-گوجه-پلوی-ایلام",
  "https://cookpad.com/ir/دستور%20غذا/17025341-کله-کنجی-ایلام",
  "https://cookpad.com/ir/دستور%20غذا/11945806-بژی-برساق-ایلامی?ref=search&search_term=برساق+(بژی+برساق)",
  "https://cookpad.com/ir/دستور%20غذا/17160050-حلوا‌ی-بگلحلوا‌ی-ایلام?ref=search&search_term=حلوا‌ی+بگل",
  "https://cookpad.com/ir/دستور%20غذا/14417562-شلکینه",
  "https://cookpad.com/ir/دستور%20غذا/16957986-چزنگ-رغو-نان-محلی-استان-ایلام?ref=search&search_term=شلکینه",
  "https://cookpad.com/ir/دستور%20غذا/17071129-خورشت-خرفه-ایلام",
  "https://cookpad.com/ir/دستور%20غذا/15860141-شیرینی-کعک-هلی-ایلام"
],
        "لرستان": [
  "https://cookpad.com/ir/دستور%20غذا/24665899-زرشک-پلو-به-سبک-لرستان",
  "https://cookpad.com/ir/دستور%20غذا/24085292-آش-رشته-لرستان?ref=search&search_term=لرستان",
  "https://cookpad.com/ir/دستور%20غذا/24463184-کباب-کوبیده-لرستان",
  "https://cookpad.com/ir/دستور%20غذا/24463161-کشکینه-محلی-لرستان",
  "https://cookpad.com/ir/دستور%20غذا/24266767-جوجوشحلوا‌ی-سنتی-لرستان",
  "https://cookpad.com/ir/دستور%20غذا/24226978-کباب-شامی-لرستان?ref=search&search_term=لرستان",
  "https://cookpad.com/ir/دستور%20غذا/24091797-زیربرنجی-لرستان?ref=search&search_term=لرستان",
  "https://cookpad.com/ir/دستور%20غذا/24085856-نان-محلی-لرستان?ref=search&search_term=لرستان",
  "https://cookpad.com/ir/دستور%20غذا/24061258-حلیم-گندم-لرستان",
  "https://cookpad.com/ir/دستور%20غذا/17201730-شب-یلدا-شو-چله-لرستان",
  "https://cookpad.com/ir/دستور%20غذا/17196337-گنم-شادونه-لرستان-مخصوص-شو-چلهگندم-شاهدونه-مخصوص-یلدا",
  "https://cookpad.com/ir/دستور%20غذا/17185225-چزنگ-رغومیان-وعده-محلی-استان-لرستان",
  "https://cookpad.com/ir/دستور%20غذا/17123365-ماست-کز-زده-دودی-لرستان",
  "https://cookpad.com/ir/دستور%20غذا/17121926-نان-کلوَا-لرستان",
  "https://cookpad.com/ir/دستور%20غذا/17085940-نان-چزنگ-رغو-لرستانشلکینه?ref=search&search_term=لرستان",
  "https://cookpad.com/ir/دستور%20غذا/16959159-نان-کلوای-لرستان?ref=search&search_term=لرستان",
  "https://cookpad.com/ir/دستور%20غذا/17208889-آبگوشت-کشک-آبگوشت-دودار?ref=search&search_term=دودار",
  "https://cookpad.com/ir/دستور%20غذا/6289919-آش-ترخینه-دوغ-یا-کشکینه-لرستان",
  "https://cookpad.com/ir/دستور%20غذا/24339657-آش-بادمجان?ref=search&search_term=آش+بادمجان",
  "https://cookpad.com/ir/%D8%AF%D8%B3%D8%AA%D9%88%D8%B1%20%D8%BA%D8%B0%D8%A7/14203012-%D8%A2%D8%B4-%D8%B4%D9%84%D9%87-%D8%A2%D8%B4-%D9%85%D8%AD%D9%84%DB%8C-%D8%AE%D8%B1%D9%85-%D8%A2%D8%A8%D8%A7%D8%AF"          
],
        "خوزستان": [
  "https://cookpad.com/ir/دستور%20غذا/15980262-قلیه-دزفولی-گوشت",
  "https://cookpad.com/ir/دستور%20غذا/13282371-خورشت-بامیه-خوزستانی",
  "https://cookpad.com/ir/دستور%20غذا/8191956-امگشت",
  "https://cookpad.com/ir/دستور%20غذا/14540609-توله-عربی-توله-یا-پنیرک",
  "https://cookpad.com/ir/دستور%20غذا/12967859-پلو-شوشتر‌ی-خوزستان",
  "https://cookpad.com/ir/دستور%20غذا/12898983-فلافل-آبادانی",
  "https://cookpad.com/ir/دستور%20غذا/16750156-سمبوسه-خوزستانی",
  "https://cookpad.com/ir/دستور%20غذا/16910923-پاکوره-جنوبی-پاکورا",
  "https://cookpad.com/ir/دستور%20غذا/15688495-باقالی-روغن-باگله-بالدهن",
  "https://cookpad.com/ir/دستور%20غذا/17013944-خوراک-میگو-آبادانی",
  "https://cookpad.com/ir/دستور%20غذا/13934998-ماهی-صبور-خوزستان",
  "https://chishi.ir/1321-morasa-polo/",
  "https://cookpad.com/ir/دستور%20غذا/14509640-اشکنه-دزفولی",
  "https://cookpad.com/ir/دستور%20غذا/14524617-نان-چرب-شوشتر‌ی",
  "https://cookpad.com/ir/دستور%20غذا/16803502-کلوچه-عربی-زنجبیلی",
  "https://cookpad.com/ir/دستور%20غذا/17221003-حلوا-خرمای-خشک-خوزستانی",
  "https://cookpad.com/ir/دستور%20غذا/24124091-مفتح-مفتح-غذای-معروف-خوزستان",
  "https://cookpad.com/ir/دستور%20غذا/17119527-کلوچه-خوزستانکلوچه-عربی",
  "https://cookpad.com/ir/دستور%20غذا/16748183-تثخانه-غذای-سنتی-خوزستان",
  "https://cookpad.com/ir/دستور%20غذا/16629375-مطبگ-مرغ-غذای-محلی-استان-خوزستان-ابادان-خرمشهر",
  "https://cookpad.com/ir/دستور%20غذا/15835868-لگیمات-شیرینی-سنتی-خوزستان",
  "https://cookpad.com/ir/دستور%20غذا/14156453-قرمه-سبزی-خوزستان",
  "https://cookpad.com/ir/دستور%20غذا/14075568-حمص-غذای-عربی-خوزستان",
  "https://cookpad.com/ir/دستور%20غذا/13359439-نان-کپو-فارس-نان-پیتە-خوزستان",
  "https://cookpad.com/ir/دستور%20غذا/12981114-رنگینک-خوزستان",
  "https://cookpad.com/ir/دستور%20غذا/12981114-رنگینک-خوزستان",
  "https://cookpad.com/ir/دستور%20غذا/12983843-آو-تفتاله-غذای-محلی-بختیاریخوزستان",
  "https://cookpad.com/ir/دستور%20غذا/10431534-شیرینی-سنتی-خوزستان-کلیت-عرب",
  "https://cookpad.com/ir/دستور%20غذا/11345430-آش-خوزستان"
],
        "همدان": [
  "https://cookpad.com/ir/دستور%20غذا/16232848-آبگوشت-کلم-قمری",
  "https://cookpad.com/ir/دستور%20غذا/23966091-کباب-سرداشی-همدان",
  "https://cookpad.com/ir/دستور%20غذا/12948597-آبگوشت-قورمه-همدان",
  "https://cookpad.com/ir/دستور%20غذا/12944661-کوفته-همدان?ref=search&search_term=همدان",
  "https://cookpad.com/ir/دستور%20غذا/17159604-آش-هویج-و-جو-همدانی",
  "https://cookpad.com/ir/دستور%20غذا/11115948-آش-پتله",
  "https://cookpad.com/ir/دستور%20غذا/17050317-ترحلوا‌ی-هویج",
  "https://cookpad.com/ir/دستور%20غذا/24611991-نان-روغنی-همدان-اگر%D8%AF%DA%A9?ref=search&search_term=همدان",
  "https://cookpad.com/ir/دستور%20غذا/24505363-ترش-شوربا-آش-ترش-همدان",
  "https://cookpad.com/ir/دستور%20غذا/24506267-آش-چغندر-همدان-چغندر-سفید",
  "https://cookpad.com/ir/دستور%20غذا/24414021-کباب-دیزی-همدان",
  "https://cookpad.com/ir/دستور%20غذا/24307874-اگر%D8%AF%DA%A9-همدان",
  "https://cookpad.com/ir/دستور%20غذا/16931501-کاچی-همدان-بدون-شکر",
  "https://cookpad.com/ir/دستور%20غذا/17227562-انگشت-پیچ-همدان",
  "https://cookpad.com/ir/دستور%20غذا/17286341-قووت-همدان-دیمـاج",
  "https://cookpad.com/ir/دستور%20غذا/17169553-حلوا-تخم-مرغی-همدانبدون-آرد",
  "https://cookpad.com/ir/دستور%20غذا/17078680-نان-کماج-همدان-با-آون-توستربدون-ورزبـدون-استراحت",
  "https://cookpad.com/ir/دستور%20غذا/16982378-نان-گرده-همدان?ref=search&search_term=همدان",
  "https://cookpad.com/ir/دستور%20غذا/16636369-نان-شیرمال-همدانی",
  "https://cookpad.com/ir/دستور%20غذا/16385141-آبگوشت-فلفل-همدان",
  "https://cookpad.com/ir/دستور%20غذا/13753774-آش-ترخینه-همدانی",
  "https://cookpad.com/ir/دستور%20غذا/13442756-آش-خیار-چنبر-همدان",
  "https://cookpad.com/ir/دستور%20غذا/12976787-آبگوشت-بادمجان-همدان",
  "https://cookpad.com/ir/دستور%20غذا/12967897-رشته-پلو-مرغ-همدان",
  "https://cookpad.com/ir/دستور%20غذا/12922990-بیرساق-محلی-همدان",
  "https://cookpad.com/ir/دستور%20غذا/11202063-آش-دوغ-همدان",
  "https://cookpad.com/ir/دستور%20غذا/3975271-کباب-همدانی?ref=search&search_term=هویج+همدان"
],
        "مرکزی": [
  "https://cookpad.com/ir/دستور%20غذا/17208889-آبگوشت-کشک-آبگوشت-دودار?ref=search&search_term=دودار",
  "https://cookpad.com/ir/دستور%20غذا/13520688-آش-جو-و-شکمبه?ref=search&search_term=آش+جو+با+شکمبه",
  "https://cookpad.com/ir/دستور%20غذا/15030928-آش-خیار-چنبر?ref=search&search_term=آش+خیار+چنبر",
  "https://cookpad.com/ir/دستور%20غذا/22672737-کلهجوش?ref=search&search_term=کله‌جوش",
  "https://cookpad.com/ir/دستور%20غذا/24085539-رشته-پلو?ref=search&search_term=رشته‌پلو",
  "https://cookpad.com/ir/دستور%20غذا/17215079-کباب-تتالی-اراکی?ref=search&search_term=کباب+تتالی",
  "https://cookpad.com/ir/دستور%20غذا/24141761-شفتە-اراکی?ref=search&search_term=شفتە+اراکی",
  "https://cookpad.com/ir/دستور%20غذا/16326681-کوفته-کشک?ref=search&search_term=کوفته+کشک",
  "https://cookpad.com/ir/دستور%20غذا/7222961-گورماست?ref=search&search_term=گورماست",
  "https://cookpad.com/ir/دستور%20غذا/24336635-آش-انار?ref=search&search_term=آش+انار",
  "https://cookpad.com/ir/دستور%20غذا/17143272-آش-آلو?ref=search&search_term=آش+آلو",
  "http://cookpad.com/ir/دستور%20غذا/14843377-دملمه",
  "https://cookpad.com/ir/دستور%20غذا/12790611-نان-فطیر-اراکی?ref=search&search_term=نان+فطیر+اراکی",
  "https://cookpad.com/ir/دستور%20غذا/24514888-آش-بی-بی-سه-شنبه-اراکی?ref=search&search_term=آش+نذری+اراکی",
  "https://cookpad.com/ir/دستور%20غذا/10971840-آش-لبو-اراک?ref=search&search_term=اراک",
  "https://cookpad.com/ir/دستور%20غذا/9700483-چیلیک-اراک?ref=search&search_term=اراک",
  "https://cookpad.com/ir/دستور%20غذا/8672123-یتیمچه-غذای-مخصوص-اراک?ref=search&search_term=اراک",
  "https://cookpad.com/ir/دستور%20غذا/16200364-گوشفیل-اراکی?ref=search&search_term=اراکی",
  "https://cookpad.com/ir/دستور%20غذا/14971102-حلوا‌ی-اراکی-سریع-و-ساده?ref=search&search_term=اراکی",
  "https://cookpad.com/ir/دستور%20غذا/14177410-آش-جودوغ-اراکی?ref=search&search_term=اراکی",
  "https://cookpad.com/ir/دستور%20غذا/12552780-خاماتو-اراکی?ref=search&search_term=اراکی",
  "https://cookpad.com/ir/دستور%20غذا/11473233-برساق-اراکی?ref=search&search_term=اراکی",
  "https://cookpad.com/ir/دستور%20غذا/16973714-سماق-پلو-با-آبگوشت-سفید-ساوه-ای?ref=search&search_term=ساوه",
  "https://cookpad.com/ir/دستور%20غذا/17079913-آش-جو-ساوه?ref=search&search_term=ساوه",
  "https://cookpad.com/ir/دستور%20غذا/16973716-آبگوشت-سفید-آبگوشت-ساوه?ref=search&search_term=ساوه",
  "https://cookpad.com/ir/دستور%20غذا/16084351-نان-قندی-یا-قطاب-ساوه-ای?ref=search&search_term=ساوه",
  "https://cookpad.com/ir/دستور%20غذا/24505399-آش-مصطفی-اراکى?ref=search&search_term=ساوه",
  "https://cookpad.com/ir/دستور%20غذا/14335969-دسر-کشک-و-لبو?ref=search&search_term=ساوه",
  "https://cookpad.com/ir/دستور%20غذا/16955402-باسلوق-شیره-انگور-با-گردو?ref=search&search_term=ساوه",
  "https://cookpad.com/ir/دستور%20غذا/11269291-نون-چایی-توتکی-نان-محلی-شهر-ساوه?ref=search&search_term=ساوه",
  "https://cookpad.com/ir/دستور%20غذا/5530126-خورش-آلو-نعنا?ref=search&search_term=ساوه",
  "https://cookpad.com/ir/دستور%20غذا/5220010-نون-اگر%D8%AF%DA%A9?ref=search&search_term=ساوه",
  "https://cookpad.com/ir/دستور%20غذا/4564324-روح-افزا-شیرینی-عید?ref=search&search_term=ساوه"
],
    }

    for province, urls in all_urls.items():
        if not urls:
            print(f"{province} Done :")
            continue

        recipes = process_recipes_by_province(province, urls)

        filename = f"cookpad_{province}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(recipes, f, ensure_ascii=False, indent=2)

        print(f"{province} done :) ")


if __name__ == "__main__":
    main()

کرمانشاه done :) 
ایلام done :) 
لرستان done :) 
خوزستان done :) 
همدان done :) 
مرکزی done :) 


# Crawling from Roostanet website

In [27]:
CITY_COORDINATES = {
    "لرستان": {
        "خرم‌آباد": {"city": "خرم‌آباد", "latitude": 33.4878, "longitude": 48.3558},
        "بروجرد": {"city": "بروجرد", "latitude": 33.8972, "longitude": 48.7518},
        "دورود": {"city": "دورود", "latitude": 33.4950, "longitude": 49.0236},
        "کوهدشت": {"city": "کوهدشت", "latitude": 33.5295, "longitude": 47.6136},
        "ازنا": {"city": "ازنا", "latitude": 33.4558, "longitude": 49.4356},
        "الشتر": {"city": "الشتر", "latitude": 33.8442, "longitude": 48.2689},
        "پلدختر": {"city": "پلدختر", "latitude": 33.1372, "longitude": 47.7133},
        "الیگودرز": {"city": "الیگودرز", "latitude": 33.3971, "longitude": 49.7010},
        "نورآباد": {"city": "نورآباد", "latitude": 34.0733, "longitude": 47.9725},
        "چغلوندی": {"city": "چغلوندی", "latitude": 33.6500, "longitude": 48.7000},
        "زاغه": {"city": "زاغه", "latitude": 33.5000, "longitude": 48.7167},
        "سپیددشت": {"city": "سپیددشت", "latitude": 33.1833, "longitude": 47.7667}
    },
    "خوزستان": {
        "اهواز": {"city": "اهواز", "latitude": 31.3183, "longitude": 48.6706},
        "آبادان": {"city": "آبادان", "latitude": 30.3392, "longitude": 48.3043},
        "خرمشهر": {"city": "خرمشهر", "latitude": 30.4394, "longitude": 48.1817},
        "دزفول": {"city": "دزفول", "latitude": 32.3811, "longitude": 48.4058},
        "شوشتر": {"city": "شوشتر", "latitude": 32.0450, "longitude": 48.8594},
        "مسجدسلیمان": {"city": "مسجدسلیمان", "latitude": 31.9543, "longitude": 49.2853},
        "رامهرمز": {"city": "رامهرمز", "latitude": 31.2782, "longitude": 49.5891},
        "بهبهان": {"city": "بهبهان", "latitude": 30.5953, "longitude": 50.2431},
        "اندیمشک": {"city": "اندیمشک", "latitude": 32.4600, "longitude": 48.3594},
        "شوش": {"city": "شوش", "latitude": 32.1942, "longitude": 48.2436},
        "هفتکل": {"city": "هفتکل", "latitude": 31.4453, "longitude": 49.5253},
        "لالی": {"city": "لالی", "latitude": 32.3289, "longitude": 49.0936},
        "باغ‌ملک": {"city": "باغ‌ملک", "latitude": 31.5231, "longitude": 49.8861},
        "هویزه": {"city": "هویزه", "latitude": 31.4653, "longitude": 48.0789},
        "حمیدیه": {"city": "حمیدیه", "latitude": 31.4833, "longitude": 48.4333}
    }
}

def normalize_city_name(city, province):
    city = city.strip()
    corrections = {
        "خرم آباد": "خرم‌آباد",
        "خرماباد": "خرم‌آباد",
        "آبادن": "آبادان",
        "بروجرد ": "بروجرد",
        "خرم آباد ": "خرم‌آباد"
    }
    normalized = corrections.get(city, city)
    if normalized not in CITY_COORDINATES.get(province, {}):
        return list(CITY_COORDINATES[province].keys())[0]  # پیش‌فرض: اولین شهر استان
    return normalized

def parse_food_table(table):
    data = {}
    rows = table.find_all("tr")
    for row in rows:
        cols = row.find_all("td")
        if len(cols) == 2:
            label = cols[0].get_text(strip=True).replace(":", "")
            value = cols[1].get_text("\n", strip=True).replace("\n", " ").strip()
            if label and value:
                data[label] = value
    return data

def parse_ingredients(raw_text):
    ingredients = []
    pattern = r'([\u0600-\u06FF\s\-()]+?)\s*(\d+\.?\d*)\s*(گرم|کیلوگرم|لیوان|فنجان|قاشق غذاخوری|قاشق چایخوری|قاشق|عدد|حبه|بسته|ورقه)?'
    for line in raw_text.split("\n"):
        if not line.strip():
            continue
        for match in re.finditer(pattern, line):
            name = match.group(1).strip()
            try:
                amount = float(match.group(2)) if '.' in match.group(2) else int(match.group(2))
            except ValueError:
                continue
            unit = match.group(3) or "واحد نامشخص"
            ingredients.append({"name": name, "amount": amount, "unit": unit})
    return ingredients

def split_instructions(raw_text):
    if re.search(r'\d+\.', raw_text):
        return [s.strip() for s in re.split(r'\d+\.', raw_text) if s.strip()]
    return [s.strip() for s in re.split(r'[.!؟؛]\s+', raw_text) if s.strip()]

def is_food_table(table):
    required_fields = ["نام غذا", "مواد لازم", "طرز تهیه"]
    text = table.get_text()
    return all(field in text for field in required_fields)

def detect_meal_type(title, ingredients_text):
    title_lower = title.lower()
    ingredients_lower = ingredients_text.lower() if ingredients_text else ""
    if any(w in title_lower for w in ["آبگوشت", "جگر", "خورشت", "خورش", "کباب", "کوفته", "دلمه", "کتلت", "شامی"]):
        return ["غذای اصلی گوشتی"]
    if any(w in title_lower for w in ["پلو", "برنج", "برنجی", "چلو"]):
        return ["غذای اصلی برنجی"]
    if any(w in title_lower for w in ["آش", "سوپ", "حلیم", "اشکنه"]):
        return ["آش و سوپ"]
    if any(w in title_lower for w in ["خوراک", "بورانی", "کوکو", "املت"]):
        if "گوشت" in ingredients_lower:
            return ["غذای اصلی گوشتی"]
        return ["غذای اصلی گیاهی"]
    if any(w in title_lower for w in ["شیرینی", "حلوا", "کیک", "بستنی", "دسر", "شله", "مهلبیه"]):
        return ["دسر"]
    if any(w in title_lower for w in ["نان", "ساجی", "لواش", "تافتون"]):
        return ["نان و شیرینی"]
    if any(w in title_lower for w in ["سالاد", "ترشی", "ماست"]):
        return ["پیش غذا و مکمل"]
    if "گوشت" in ingredients_lower:
        return ["غذای اصلی گوشتی"]
    if "برنج" in ingredients_lower:
        return ["غذای اصلی برنجی"]
    if "شکر" in ingredients_lower or "عسل" in ingredients_lower:
        return ["دسر"]
    return ["-"]

def detect_occasion(title, meal_type, ingredients=None, tags=None, season=None, time_of_day=None, is_weekend=False):
    title_lower = title.lower()
    MAIN_OCCASIONS = {
        "صبحانه": ["صبحانه", "ناشتا", "کله", "پاچه", "حلیم", "املت", "نیمرو", "پنیر"],
        "ناهار": ["ناهار", "چاشت", "ظهر", "پلو", "برنج", "چلو", "خورشت"],
        "شام": ["شام", "شبان", "کباب", "آبگوشت", "دیزی"],
        "عصرانه": ["عصرانه", "چای", "کیک", "شیرینی", "دسر"],
        "سحری": ["سحری", "رمضان"],
        "افطاری": ["افطاری", "افطار"]
    }
    for occ in ["سحری", "افطاری"]:
        if any(kw in title_lower for kw in MAIN_OCCASIONS[occ]):
            return occ
    for occasion, keywords in MAIN_OCCASIONS.items():
        if any(kw in title_lower for kw in keywords):
            return occasion
    if time_of_day == "صبح":
        return "صبحانه"
    if time_of_day == "عصر":
        return "عصرانه"
    return "شام" if not is_weekend else "ناهار"

def extract_all_foods(tables, province):
    all_foods = []
    for table in tables:
        if not is_food_table(table):
            continue
        try:
            raw_data = parse_food_table(table)
            raw_city = raw_data.get("شهر", "").strip()
            city_name = normalize_city_name(raw_city, province)
            coordinates = CITY_COORDINATES[province].get(city_name, {"city": city_name, "latitude": None, "longitude": None})
            ingredients_text = raw_data.get("مواد لازم", "")
            ingredients = parse_ingredients(ingredients_text)
            title = raw_data.get("نام غذا", "نامشخص").strip()
            meal_type = detect_meal_type(title, ingredients_text)
            occasion = detect_occasion(title, meal_type, ingredients_text)
            food_structured = {
                "title": title,
                "locations": [
                    {
                        "province": province,
                        "city": city_name,
                        "latitude": coordinates.get("latitude"),
                        "longitude": coordinates.get("longitude")
                    }
                ],
                "ingredients": ingredients,
                "instructions": split_instructions(raw_data.get("طرز تهیه", "")),
                "meal_type": meal_type,
                "occasion": occasion,
                "images": {
                    "main": None,
                    "steps": []
                }
            }
            all_foods.append(food_structured)

    return all_foods

def extract_foods_from_url(url, province):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")
        tables = soup.find_all("table")
        return extract_all_foods(tables, province)


PROVINCE_URLS = {
    "لرستان": ["https://roostanet.ir/fa/8635"],
    "خوزستان": ["https://roostanet.ir/fa/6118"]
}

def main():
    all_data = {}
    missing_coords = set()

    for province, urls in PROVINCE_URLS.items():
        foods = []
        for url in urls:
            foods += extract_foods_from_url(url, province)
        all_data[province] = foods

        with open(f"Roostanet_{province}.json", "w", encoding="utf-8") as f:
            json.dump(foods, f, ensure_ascii=False, indent=2)

        print(f"{province} Done :)")

        for food in foods:
            for loc in food["locations"]:
                if loc["latitude"] is None or loc["longitude"] is None:
                    missing_coords.add((province, loc["city"]))


if __name__ == "__main__":
    main()

لرستان Done :)
خوزستان Done :)


# Crawling from Canbo website

In [46]:
CITY_LOCATION_MAP = {
    "کرمانشاه": {
        "کرمانشاه": {"city": "کرمانشاه", "latitude": 34.3142, "longitude": 47.0650},
        "اسلام‌آباد غرب": {"city": "اسلام‌آباد غرب", "latitude": 34.1111, "longitude": 46.5278},
        "جوانرود": {"city": "جوانرود", "latitude": 34.8061, "longitude": 46.4889},
        "کنگاور": {"city": "کنگاور", "latitude": 34.5042, "longitude": 47.9650},
        "سرپل ذهاب": {"city": "سرپل ذهاب", "latitude": 34.4572, "longitude": 45.8611},
        "سنقر": {"city": "سنقر", "latitude": 34.7833, "longitude": 47.6000},
        "هرسین": {"city": "هرسین", "latitude": 34.2711, "longitude": 47.5861},
        "صحنه": {"city": "صحنه", "latitude": 34.4811, "longitude": 47.6833},
        "پاوه": {"city": "پاوه", "latitude": 35.0431, "longitude": 46.3561},
        "روانسر": {"city": "روانسر", "latitude": 34.7167, "longitude": 46.6500},
        "گیلانغرب": {"city": "گیلانغرب", "latitude": 34.1397, "longitude": 45.9200},
        "قصر شیرین": {"city": "قصر شیرین", "latitude": 34.5150, "longitude": 45.5772},
        "تازه‌آباد": {"city": "تازه‌آباد", "latitude": 34.7442, "longitude": 46.1511},
        "کرند غرب": {"city": "کرند غرب", "latitude": 34.2833, "longitude": 46.2333},
        "سورانه": {"city": "سورانه", "latitude": 34.4425, "longitude": 45.8431},
        "باینگان": {"city": "باینگان", "latitude": 34.9667, "longitude": 46.2833},
        "ثلاث باباجانی": {"city": "ثلاث باباجانی", "latitude": 34.7358, "longitude": 46.1494},
    },
    
    "ایلام": {
        "ایلام": {"city": "ایلام", "latitude": 33.6374, "longitude": 46.4227},
        "دهلران": {"city": "دهلران", "latitude": 32.7450, "longitude": 47.2644},
        "آبدانان": {"city": "آبدانان", "latitude": 33.1561, "longitude": 47.4328},
        "دره‌شهر": {"city": "دره‌شهر", "latitude": 33.1383, "longitude": 47.3786},
        "ایوان": {"city": "ایوان", "latitude": 33.8275, "longitude": 46.3014},
        "مهران": {"city": "مهران", "latitude": 33.1336, "longitude": 46.1839},
        "لومار": {"city": "لومار", "latitude": 33.5675, "longitude": 46.8142},
        "چرداول": {"city": "چرداول", "latitude": 33.8333, "longitude": 46.8333},
        "سرابله": {"city": "سرابله", "latitude": 33.4639, "longitude": 46.5664},
        "موسیان": {"city": "موسیان", "latitude": 33.4833, "longitude": 46.8000},
    },
    
    "لرستان": {
        "خرم‌آباد": {"city": "خرم‌آباد", "latitude": 33.4878, "longitude": 48.3558},
        "بروجرد": {"city": "بروجرد", "latitude": 33.8972, "longitude": 48.7518},
        "دورود": {"city": "دورود", "latitude": 33.4950, "longitude": 49.0236},
        "کوهدشت": {"city": "کوهدشت", "latitude": 33.5295, "longitude": 47.6136},
        "ازنا": {"city": "ازنا", "latitude": 33.4558, "longitude": 49.4356},
        "الشتر": {"city": "الشتر", "latitude": 33.8442, "longitude": 48.2689},
        "پلدختر": {"city": "پلدختر", "latitude": 33.1372, "longitude": 47.7133},
        "الیگودرز": {"city": "الیگودرز", "latitude": 33.3971, "longitude": 49.7010},
        "نورآباد": {"city": "نورآباد", "latitude": 34.0733, "longitude": 47.9725},
        "چغلوندی": {"city": "چغلوندی", "latitude": 33.6500, "longitude": 48.7000},
        "زاغه": {"city": "زاغه", "latitude": 33.5000, "longitude": 48.7167},
        "سپیددشت": {"city": "سپیددشت", "latitude": 33.1833, "longitude": 47.7667},
    },
    
    "خوزستان": {
        "اهواز": {"city": "اهواز", "latitude": 31.3183, "longitude": 48.6706},
        "آبادان": {"city": "آبادان", "latitude": 30.3392, "longitude": 48.3043},
        "خرمشهر": {"city": "خرمشهر", "latitude": 30.4394, "longitude": 48.1817},
        "دزفول": {"city": "دزفول", "latitude": 32.3811, "longitude": 48.4058},
        "شوشتر": {"city": "شوشتر", "latitude": 32.0450, "longitude": 48.8594},
        "مسجدسلیمان": {"city": "مسجدسلیمان", "latitude": 31.9543, "longitude": 49.2853},
        "رامهرمز": {"city": "رامهرمز", "latitude": 31.2782, "longitude": 49.5891},
        "بهبهان": {"city": "بهبهان", "latitude": 30.5953, "longitude": 50.2431},
        "اندیمشک": {"city": "اندیمشک", "latitude": 32.4600, "longitude": 48.3594},
        "شوش": {"city": "شوش", "latitude": 32.1942, "longitude": 48.2436},
        "هفتکل": {"city": "هفتکل", "latitude": 31.4453, "longitude": 49.5253},
        "لالی": {"city": "لالی", "latitude": 32.3289, "longitude": 49.0936},
        "باغ‌ملک": {"city": "باغ‌ملک", "latitude": 31.5231, "longitude": 49.8861},
        "هویزه": {"city": "هویزه", "latitude": 31.4653, "longitude": 48.0789},
        "حمیدیه": {"city": "حمیدیه", "latitude": 31.4833, "longitude": 48.4333},
    },
    
    "همدان": {
        "همدان": {"city": "همدان", "latitude": 34.7986, "longitude": 48.5146},
        "ملایر": {"city": "ملایر", "latitude": 34.3143, "longitude": 48.8210},
        "نهاوند": {"city": "نهاوند", "latitude": 34.2005, "longitude": 48.3758},
        "تویسرکان": {"city": "تویسرکان", "latitude": 34.5497, "longitude": 48.4497},
        "رزن": {"city": "رزن", "latitude": 35.4108, "longitude": 49.0127},
        "کبودرآهنگ": {"city": "کبودرآهنگ", "latitude": 35.2203, "longitude": 48.1856},
        "بهار": {"city": "بهار", "latitude": 34.8997, "longitude": 48.4336},
        "فامنین": {"city": "فامنین", "latitude": 35.0121, "longitude": 48.9876},
        "اسدآباد": {"city": "اسدآباد", "latitude": 34.7981, "longitude": 47.9890},
        "قهاوند": {"city": "قهاوند", "latitude": 34.7833, "longitude": 48.5833},
        "لالجین": {"city": "لالجین", "latitude": 35.0167, "longitude": 48.5167},
    },
    
    "مرکزی": {
        "اراک": {"city": "اراک", "latitude": 34.0917, "longitude": 49.6892},
        "ساوه": {"city": "ساوه", "latitude": 35.0214, "longitude": 50.3568},
        "خمین": {"city": "خمین", "latitude": 33.6414, "longitude": 50.0780},
        "محلات": {"city": "محلات", "latitude": 33.9208, "longitude": 50.4271},
        "شازند": {"city": "شازند", "latitude": 33.9356, "longitude": 49.4211},
        "تفرش": {"city": "تفرش", "latitude": 34.6936, "longitude": 50.0161},
        "کمیجان": {"city": "کمیجان", "latitude": 34.7164, "longitude": 49.3870},
        "زرندیه": {"city": "زرندیه", "latitude": 35.2139, "longitude": 50.4969},
        "آشتیان": {"city": "آشتیان", "latitude": 34.5219, "longitude": 50.0119},
        "دلیجان": {"city": "دلیجان", "latitude": 33.9906, "longitude": 50.6839},
        "خنداب": {"city": "خنداب", "latitude": 34.3833, "longitude": 49.1833},
        "فراهان": {"city": "فراهان", "latitude": 34.5094, "longitude": 49.6919},
        "مامونیه": {"city": "مامونیه", "latitude": 35.3000, "longitude": 50.5000},
    }
}

SUFFIX_LOCATION_MAP = {
    "کرمانشاهی": ("کرمانشاه", "کرمانشاه"),
    "اسلام‌آبادی": ("کرمانشاه", "اسلام‌آباد غرب"),
    "جوانرودی": ("کرمانشاه", "جوانرود"),
    "کنگاوری": ("کرمانشاه", "کنگاور"),
    "سرپل‌ذهابی": ("کرمانشاه", "سرپل ذهاب"),
    "سنقری": ("کرمانشاه", "سنقر"),
    "هرسینی": ("کرمانشاه", "هرسین"),
    "صحنه‌ای": ("کرمانشاه", "صحنه"),
    "پاوه‌ای": ("کرمانشاه", "پاوه"),
    "روانسری": ("کرمانشاه", "روانسر"),
    "گیلانغربی": ("کرمانشاه", "گیلانغرب"),
    "قصرشیرینی": ("کرمانشاه", "قصر شیرین"),
    "تازه‌آبادی": ("کرمانشاه", "تازه‌آباد"),
    "کرندی": ("کرمانشاه", "کرند غرب"),
    "سورانه‌ای": ("کرمانشاه", "سورانه"),
    "باینگانی": ("کرمانشاه", "باینگان"),
    "ثلاث‌باباجانی": ("کرمانشاه", "ثلاث باباجانی"),

    "ایلامی": ("ایلام", "ایلام"),
    "دهلرانی": ("ایلام", "دهلران"),
    "آبدانانی": ("ایلام", "آبدانان"),
    "دره‌شهری": ("ایلام", "دره‌شهر"),
    "ایوانی": ("ایلام", "ایوان"),
    "مهرانی": ("ایلام", "مهران"),
    "لوماری": ("ایلام", "لومار"),
    "چرداولی": ("ایلام", "چرداول"),
    "سرابله‌ای": ("ایلام", "سرابله"),
    "موسیانی": ("ایلام", "موسیان"),

    "خرم‌آبادی": ("لرستان", "خرم‌آباد"),
    "بروجردی": ("لرستان", "بروجرد"),
    "دورودی": ("لرستان", "دورود"),
    "کوهدشتی": ("لرستان", "کوهدشت"),
    "ازنایی": ("لرستان", "ازنا"),
    "الشتری": ("لرستان", "الشتر"),
    "پلدختری": ("لرستان", "پلدختر"),
    "الیگودرزی": ("لرستان", "الیگودرز"),
    "نورآبادی": ("لرستان", "نورآباد"),
    "چغلوندی": ("لرستان", "چغلوندی"),
    "زاغه‌ای": ("لرستان", "زاغه"),
    "سپیددشتی": ("لرستان", "سپیددشت"),

    "اهوازی": ("خوزستان", "اهواز"),
    "آبادانی": ("خوزستان", "آبادان"),
    "خرمشهری": ("خوزستان", "خرمشهر"),
    "دزفولی": ("خوزستان", "دزفول"),
    "شوشتری": ("خوزستان", "شوشتر"),
    "مسجدسلیمانی": ("خوزستان", "مسجدسلیمان"),
    "رامهرمزی": ("خوزستان", "رامهرمز"),
    "بهبهانی": ("خوزستان", "بهبهان"),
    "اندیمشکی": ("خوزستان", "اندیمشک"),
    "شوشی": ("خوزستان", "شوش"),
    "هفتکلی": ("خوزستان", "هفتکل"),
    "لالی": ("خوزستان", "لالی"),
    "باغ‌ملکی": ("خوزستان", "باغ‌ملک"),
    "هویزه‌ای": ("خوزستان", "هویزه"),
    "حمیدیه‌ای": ("خوزستان", "حمیدیه"),

    "همدانی": ("همدان", "همدان"),
    "ملایری": ("همدان", "ملایر"),
    "نهاوندی": ("همدان", "نهاوند"),
    "تویسرکانی": ("همدان", "تویسرکان"),
    "رزنی": ("همدان", "رزن"),
    "کبودرآهنگی": ("همدان", "کبودرآهنگ"),
    "بهاری": ("همدان", "بهار"),
    "فامنینی": ("همدان", "فامنین"),
    "اسدآبادی": ("همدان", "اسدآباد"),
    "قهاوندی": ("همدان", "قهاوند"),
    "لالجینی": ("همدان", "لالجین"),

    "اراکی": ("مرکزی", "اراک"),
    "ساوه‌ای": ("مرکزی", "ساوه"),
    "خمینی": ("مرکزی", "خمین"),
    "محلاتی": ("مرکزی", "محلات"),
    "شازندی": ("مرکزی", "شازند"),
    "تفرشی": ("مرکزی", "تفرش"),
    "کمیجانی": ("مرکزی", "کمیجان"),
    "زرندیه‌ای": ("مرکزی", "زرندیه"),
    "آشتیانی": ("مرکزی", "آشتیان"),
    "دلیجانی": ("مرکزی", "دلیجان"),
    "خندابی": ("مرکزی", "خنداب"),
    "فراهانی": ("مرکزی", "فراهان"),
    "مامونیه‌ای": ("مرکزی", "مامونیه"),
}

def normalize_city(city, province):
    if not city:
        return list(CITY_LOCATION_MAP.get(province, {}).keys())[0] if CITY_LOCATION_MAP.get(province) else None
    
    city = city.strip()
    
    replacements = {
        "خرم آباد": "خرم‌آباد",
        "خرماباد": "خرم‌آباد",
        "اسلام آباد": "اسلام‌آباد غرب",
        "قصرشیرین": "قصر شیرین",
        "سرپل ذهاب": "سرپل‌ذهاب",
        "سرپلذهاب": "سرپل‌ذهاب",
        "باغ ملک": "باغ‌ملک",
    }
    
    city = replacements.get(city, city)
    
    if province in CITY_LOCATION_MAP and city in CITY_LOCATION_MAP[province]:
        return city
    
    if province in CITY_LOCATION_MAP and CITY_LOCATION_MAP[province]:
        return list(CITY_LOCATION_MAP[province].keys())[0]
    
    return city


def extract_province_and_city_from_url(url, province_group=None):
    try:
        path = urlparse(url).path
        last_part = unquote(path.split('/')[-2])
        cleaned = re.sub(r'[^آ-یa-zA-Z]', '', last_part)

        for suffix, (province, city) in SUFFIX_LOCATION_MAP.items():
            if suffix in cleaned:
                return province, city

        for province, cities in CITY_LOCATION_MAP.items():
            if province in cleaned:
                for city in cities:
                    if city in cleaned:
                        return province, city
                return province, list(cities.keys())[0]

        if province_group and province_group in CITY_LOCATION_MAP:
            return province_group, list(CITY_LOCATION_MAP[province_group].keys())[0]

    except Exception as e:
        print(f"خطا در استخراج استان و شهر: {e}")

    return "نامشخص", "نامشخص"

def parse_ingredients(raw_html):
    ingredients = []
    for content in raw_html.select(".recipe-materials .mat-content"):
        name_tag = content.find("div", class_="fw-bold")
        amount_tag = content.find("div", class_="small")
        name = name_tag.text.strip() if name_tag else ""
        amount_unit = amount_tag.text.strip() if amount_tag else ""
        
        pattern = r'([\d٫\.\/\d]+)?\s*(ق\.?غ|ق\.?م|ق\.?چ|لیوان|فنجان|عدد|گرم|بسته|حبه|ورق|پیمانه|کاسه|بشقاب|قاشق غذاخوری|قاشق چای‌خوری|قاشق سوپخوری|قاشق مرباخوری)?'
        match = re.match(pattern, amount_unit)
        
        if match:
            amount = match.group(1).replace("٫", ".") if match.group(1) else None
            try:
                amount = float(amount) if amount else None
            except:
                pass
            unit = match.group(2) or None
        else:
            amount, unit = None, None
            
        ingredients.append({"name": name, "amount": amount, "unit": unit})
    return ingredients

def split_instructions(soup):
    instructions = []
    for step in soup.select(".recipe .text-justify"):
        for p in step.find_all(["p", "li"]):
            txt = p.get_text(strip=True)
            if txt and len(txt) > 5:
                instructions.append(txt)
    return instructions

def detect_meal_type(title, ingredients_text):
    text = f"{title} {ingredients_text}".lower()
    meal_types = {
        "غذای اصلی گوشتی": ["آبگوشت", "جگر", "کباب", "خورش", "کوفته", "دلمه"],
        "غذای اصلی برنجی": ["پلو", "برنج", "چلو"],
        "آش و سوپ": ["آش", "سوپ", "حلیم"],
        "دسر": ["شیرینی", "حلوا", "کیک", "دسر"],
        "نان و شیرینی": ["نان", "ساجی", "لواش"],
        "پیش غذا": ["سالاد", "ترشی", "ماست"]
    }
    
    for meal_type, keywords in meal_types.items():
        if any(keyword in text for keyword in keywords):
            return [meal_type]
    
    return ["غذای اصلی"]

def extract_recipe_from_canbo(url, province_group=None):
    try:
        res = requests.get(url, timeout=10)
        soup = BeautifulSoup(res.content, "html.parser")
        
        province, city = extract_province_and_city_from_url(url, province_group)
        norm_city = normalize_city(city, province)
        coord = CITY_LOCATION_MAP.get(province, {}).get(norm_city, {})

        title_tag = soup.find("h1")
        title = title_tag.get_text(strip=True).replace("دستور تهیه", "") if title_tag else "نامشخص"

        image_div = soup.find("div", class_="recipe image-box")
        style = image_div.get("style", "") if image_div else ""
        image_url = re.search(r"url\('(.+?)'\)", style)
        final_image = image_url.group(1) if image_url else ""

        ingredients = parse_ingredients(soup)
        instructions = split_instructions(soup)
        meal_type = detect_meal_type(title, " ".join([i["name"] for i in ingredients]))
        occasion = ["ناهار"]  

        return {
            "title": title,
            "locations": [{
                "province": province,
                "city": norm_city,
                "latitude": coord.get("latitude"),
                "longitude": coord.get("longitude")
            }],
            "ingredients": ingredients,
            "instructions": instructions,
            "meal_type": meal_type,
            "occasion": occasion,
            "images": {
                "main": final_image,
                "steps": []
            }
        }
    except Exception as e:
        print(f"خطا در پردازش URL {url}: {e}")
        return None

if __name__ == "__main__":
    urls = {

        "ایلام": [
            "https://mag.canbo.ir/recipe/ashe-masoa-2/",
            "https://mag.canbo.ir/recipe/ashe-peroshke/",
            "https://mag.canbo.ir/recipe/kofteh-sirabi/",
            "https://mag.canbo.ir/recipe/%D9%85%DA%A9%D8%B4-%D8%A7%DB%8C%D9%84%D8%A7%D9%85%DB%8C/",
            "https://mag.canbo.ir/recipe/%DA%A9%D8%A8%D9%87-%D8%A7%DB%8C%D9%84%D8%A7%D9%85%DB%8C/",
            "https://mag.canbo.ir/recipe/ashe-peroshke/",
            "https://mag.canbo.ir/recipe/%d8%a8%d8%a7%d8%a8%d8%a7%db%8c%d8%a7%d8%b1%db%8c/",
            "https://mag.canbo.ir/recipe/ash-shorba/",
        ]
    }

    for province, province_urls in urls.items():
        recipes = []
        for url in province_urls:
            recipe = extract_recipe_from_canbo(url, province)
            if recipe:
                recipes.append(recipe)
        
        if recipes:
            filename = f"canbo_{province}.json"
            with open(filename, "w", encoding="utf-8") as f:
                json.dump(recipes, f, ensure_ascii=False, indent=2)
            print(f"{province} Done :)")

ایلام Done :)


# Crawling from Chi Shi website

In [50]:
import requests
from bs4 import BeautifulSoup
import re
import json
from urllib.parse import urlparse, unquote

WORD_TO_NUMBER = {
    "نیم": 0.5,
    "نصف": 0.5,
    "ربع": 0.25,
    "یک": 1,
    "دو": 2,
    "سه": 3,
    "چهار": 4,
    "پنج": 5,
    "شش": 6,
    "هفت": 7,
    "هشت": 8,
    "نه": 9,
    "ده": 10,
    "½": 0.5,
    "¾": 0.75,
    "¼": 0.25,
    "⅓": 0.33,
    "1/2": 0.5,
    "1/4": 0.25,
    "1/3": 0.33,
    "2/3": 0.66,
    "3/4": 0.75,
}

CITY_LOCATION_MAP = {
    "کرمانشاه": {
        "کرمانشاه": {"city": "کرمانشاه", "latitude": 34.3142, "longitude": 47.0650},
        "اسلام‌آباد غرب": {"city": "اسلام‌آباد غرب", "latitude": 34.1111, "longitude": 46.5278},
        "جوانرود": {"city": "جوانرود", "latitude": 34.8061, "longitude": 46.4889},
        "کنگاور": {"city": "کنگاور", "latitude": 34.5042, "longitude": 47.9650},
        "سرپل ذهاب": {"city": "سرپل ذهاب", "latitude": 34.4572, "longitude": 45.8611},
        "سنقر": {"city": "سنقر", "latitude": 34.7833, "longitude": 47.6000},
        "هرسین": {"city": "هرسین", "latitude": 34.2711, "longitude": 47.5861},
        "صحنه": {"city": "صحنه", "latitude": 34.4811, "longitude": 47.6833},
        "پاوه": {"city": "پاوه", "latitude": 35.0431, "longitude": 46.3561},
        "روانسر": {"city": "روانسر", "latitude": 34.7167, "longitude": 46.6500},
        "گیلانغرب": {"city": "گیلانغرب", "latitude": 34.1397, "longitude": 45.9200},
        "قصر شیرین": {"city": "قصر شیرین", "latitude": 34.5150, "longitude": 45.5772},
        "تازه‌آباد": {"city": "تازه‌آباد", "latitude": 34.7442, "longitude": 46.1511},
        "کرند غرب": {"city": "کرند غرب", "latitude": 34.2833, "longitude": 46.2333},
        "سورانه": {"city": "سورانه", "latitude": 34.4425, "longitude": 45.8431},
        "باینگان": {"city": "باینگان", "latitude": 34.9667, "longitude": 46.2833},
        "ثلاث باباجانی": {"city": "ثلاث باباجانی", "latitude": 34.7358, "longitude": 46.1494},
    },
    
    "ایلام": {
        "ایلام": {"city": "ایلام", "latitude": 33.6374, "longitude": 46.4227},
        "دهلران": {"city": "دهلران", "latitude": 32.7450, "longitude": 47.2644},
        "آبدانان": {"city": "آبدانان", "latitude": 33.1561, "longitude": 47.4328},
        "دره‌شهر": {"city": "دره‌شهر", "latitude": 33.1383, "longitude": 47.3786},
        "ایوان": {"city": "ایوان", "latitude": 33.8275, "longitude": 46.3014},
        "مهران": {"city": "مهران", "latitude": 33.1336, "longitude": 46.1839},
        "لومار": {"city": "لومار", "latitude": 33.5675, "longitude": 46.8142},
        "چرداول": {"city": "چرداول", "latitude": 33.8333, "longitude": 46.8333},
        "سرابله": {"city": "سرابله", "latitude": 33.4639, "longitude": 46.5664},
        "موسیان": {"city": "موسیان", "latitude": 33.4833, "longitude": 46.8000},
    },
    
    "لرستان": {
        "خرم‌آباد": {"city": "خرم‌آباد", "latitude": 33.4878, "longitude": 48.3558},
        "بروجرد": {"city": "بروجرد", "latitude": 33.8972, "longitude": 48.7518},
        "دورود": {"city": "دورود", "latitude": 33.4950, "longitude": 49.0236},
        "کوهدشت": {"city": "کوهدشت", "latitude": 33.5295, "longitude": 47.6136},
        "ازنا": {"city": "ازنا", "latitude": 33.4558, "longitude": 49.4356},
        "الشتر": {"city": "الشتر", "latitude": 33.8442, "longitude": 48.2689},
        "پلدختر": {"city": "پلدختر", "latitude": 33.1372, "longitude": 47.7133},
        "الیگودرز": {"city": "الیگودرز", "latitude": 33.3971, "longitude": 49.7010},
        "نورآباد": {"city": "نورآباد", "latitude": 34.0733, "longitude": 47.9725},
        "چغلوندی": {"city": "چغلوندی", "latitude": 33.6500, "longitude": 48.7000},
        "زاغه": {"city": "زاغه", "latitude": 33.5000, "longitude": 48.7167},
        "سپیددشت": {"city": "سپیددشت", "latitude": 33.1833, "longitude": 47.7667},
    },
    
    "خوزستان": {
        "اهواز": {"city": "اهواز", "latitude": 31.3183, "longitude": 48.6706},
        "آبادان": {"city": "آبادان", "latitude": 30.3392, "longitude": 48.3043},
        "خرمشهر": {"city": "خرمشهر", "latitude": 30.4394, "longitude": 48.1817},
        "دزفول": {"city": "دزفول", "latitude": 32.3811, "longitude": 48.4058},
        "شوشتر": {"city": "شوشتر", "latitude": 32.0450, "longitude": 48.8594},
        "مسجدسلیمان": {"city": "مسجدسلیمان", "latitude": 31.9543, "longitude": 49.2853},
        "رامهرمز": {"city": "رامهرمز", "latitude": 31.2782, "longitude": 49.5891},
        "بهبهان": {"city": "بهبهان", "latitude": 30.5953, "longitude": 50.2431},
        "اندیمشک": {"city": "اندیمشک", "latitude": 32.4600, "longitude": 48.3594},
        "شوش": {"city": "شوش", "latitude": 32.1942, "longitude": 48.2436},
        "هفتکل": {"city": "هفتکل", "latitude": 31.4453, "longitude": 49.5253},
        "لالی": {"city": "لالی", "latitude": 32.3289, "longitude": 49.0936},
        "باغ‌ملک": {"city": "باغ‌ملک", "latitude": 31.5231, "longitude": 49.8861},
        "هویزه": {"city": "هویزه", "latitude": 31.4653, "longitude": 48.0789},
        "حمیدیه": {"city": "حمیدیه", "latitude": 31.4833, "longitude": 48.4333},
    },
    
    "همدان": {
        "همدان": {"city": "همدان", "latitude": 34.7986, "longitude": 48.5146},
        "ملایر": {"city": "ملایر", "latitude": 34.3143, "longitude": 48.8210},
        "نهاوند": {"city": "نهاوند", "latitude": 34.2005, "longitude": 48.3758},
        "تویسرکان": {"city": "تویسرکان", "latitude": 34.5497, "longitude": 48.4497},
        "رزن": {"city": "رزن", "latitude": 35.4108, "longitude": 49.0127},
        "کبودرآهنگ": {"city": "کبودرآهنگ", "latitude": 35.2203, "longitude": 48.1856},
        "بهار": {"city": "بهار", "latitude": 34.8997, "longitude": 48.4336},
        "فامنین": {"city": "فامنین", "latitude": 35.0121, "longitude": 48.9876},
        "اسدآباد": {"city": "اسدآباد", "latitude": 34.7981, "longitude": 47.9890},
        "قهاوند": {"city": "قهاوند", "latitude": 34.7833, "longitude": 48.5833},
        "لالجین": {"city": "لالجین", "latitude": 35.0167, "longitude": 48.5167},
    },
    
    "مرکزی": {
        "اراک": {"city": "اراک", "latitude": 34.0917, "longitude": 49.6892},
        "ساوه": {"city": "ساوه", "latitude": 35.0214, "longitude": 50.3568},
        "خمین": {"city": "خمین", "latitude": 33.6414, "longitude": 50.0780},
        "محلات": {"city": "محلات", "latitude": 33.9208, "longitude": 50.4271},
        "شازند": {"city": "شازند", "latitude": 33.9356, "longitude": 49.4211},
        "تفرش": {"city": "تفرش", "latitude": 34.6936, "longitude": 50.0161},
        "کمیجان": {"city": "کمیجان", "latitude": 34.7164, "longitude": 49.3870},
        "زرندیه": {"city": "زرندیه", "latitude": 35.2139, "longitude": 50.4969},
        "آشتیان": {"city": "آشتیان", "latitude": 34.5219, "longitude": 50.0119},
        "دلیجان": {"city": "دلیجان", "latitude": 33.9906, "longitude": 50.6839},
        "خنداب": {"city": "خنداب", "latitude": 34.3833, "longitude": 49.1833},
        "فراهان": {"city": "فراهان", "latitude": 34.5094, "longitude": 49.6919},
        "مامونیه": {"city": "مامونیه", "latitude": 35.3000, "longitude": 50.5000},
    }
}

# دیکشنری پسوندهای شهرها
SUFFIX_LOCATION_MAP = {
    "کرمانشاهی": ("کرمانشاه", "کرمانشاه"),
    "اسلام‌آبادی": ("کرمانشاه", "اسلام‌آباد غرب"),
    "جوانرودی": ("کرمانشاه", "جوانرود"),
    "کنگاوری": ("کرمانشاه", "کنگاور"),
    "سرپل‌ذهابی": ("کرمانشاه", "سرپل ذهاب"),
    "سنقری": ("کرمانشاه", "سنقر"),
    "هرسینی": ("کرمانشاه", "هرسین"),
    "صحنه‌ای": ("کرمانشاه", "صحنه"),
    "پاوه‌ای": ("کرمانشاه", "پاوه"),
    "روانسری": ("کرمانشاه", "روانسر"),
    "گیلانغربی": ("کرمانشاه", "گیلانغرب"),
    "قصرشیرینی": ("کرمانشاه", "قصر شیرین"),
    "تازه‌آبادی": ("کرمانشاه", "تازه‌آباد"),
    "کرندی": ("کرمانشاه", "کرند غرب"),
    "سورانه‌ای": ("کرمانشاه", "سورانه"),
    "باینگانی": ("کرمانشاه", "باینگان"),
    "ثلاث‌باباجانی": ("کرمانشاه", "ثلاث باباجانی"),

    "ایلامی": ("ایلام", "ایلام"),
    "دهلرانی": ("ایلام", "دهلران"),
    "آبدانانی": ("ایلام", "آبدانان"),
    "دره‌شهری": ("ایلام", "دره‌شهر"),
    "ایوانی": ("ایلام", "ایوان"),
    "مهرانی": ("ایلام", "مهران"),
    "لوماری": ("ایلام", "لومار"),
    "چرداولی": ("ایلام", "چرداول"),
    "سرابله‌ای": ("ایلام", "سرابله"),
    "موسیانی": ("ایلام", "موسیان"),

    "خرم‌آبادی": ("لرستان", "خرم‌آباد"),
    "بروجردی": ("لرستان", "بروجرد"),
    "دورودی": ("لرستان", "دورود"),
    "کوهدشتی": ("لرستان", "کوهدشت"),
    "ازنایی": ("لرستان", "ازنا"),
    "الشتری": ("لرستان", "الشتر"),
    "پلدختری": ("لرستان", "پلدختر"),
    "الیگودرزی": ("لرستان", "الیگودرز"),
    "نورآبادی": ("لرستان", "نورآباد"),
    "چغلوندی": ("لرستان", "چغلوندی"),
    "زاغه‌ای": ("لرستان", "زاغه"),
    "سپیددشتی": ("لرستان", "سپیددشت"),

    "اهوازی": ("خوزستان", "اهواز"),
    "آبادانی": ("خوزستان", "آبادان"),
    "خرمشهری": ("خوزستان", "خرمشهر"),
    "دزفولی": ("خوزستان", "دزفول"),
    "شوشتری": ("خوزستان", "شوشتر"),
    "مسجدسلیمانی": ("خوزستان", "مسجدسلیمان"),
    "رامهرمزی": ("خوزستان", "رامهرمز"),
    "بهبهانی": ("خوزستان", "بهبهان"),
    "اندیمشکی": ("خوزستان", "اندیمشک"),
    "شوشی": ("خوزستان", "شوش"),
    "هفتکلی": ("خوزستان", "هفتکل"),
    "لالی": ("خوزستان", "لالی"),
    "باغ‌ملکی": ("خوزستان", "باغ‌ملک"),
    "هویزه‌ای": ("خوزستان", "هویزه"),
    "حمیدیه‌ای": ("خوزستان", "حمیدیه"),

    "همدانی": ("همدان", "همدان"),
    "ملایری": ("همدان", "ملایر"),
    "نهاوندی": ("همدان", "نهاوند"),
    "تویسرکانی": ("همدان", "تویسرکان"),
    "رزنی": ("همدان", "رزن"),
    "کبودرآهنگی": ("همدان", "کبودرآهنگ"),
    "بهاری": ("همدان", "بهار"),
    "فامنینی": ("همدان", "فامنین"),
    "اسدآبادی": ("همدان", "اسدآباد"),
    "قهاوندی": ("همدان", "قهاوند"),
    "لالجینی": ("همدان", "لالجین"),

    "اراکی": ("مرکزی", "اراک"),
    "ساوه‌ای": ("مرکزی", "ساوه"),
    "خمینی": ("مرکزی", "خمین"),
    "محلاتی": ("مرکزی", "محلات"),
    "شازندی": ("مرکزی", "شازند"),
    "تفرشی": ("مرکزی", "تفرش"),
    "کمیجانی": ("مرکزی", "کمیجان"),
    "زرندیه‌ای": ("مرکزی", "زرندیه"),
    "آشتیانی": ("مرکزی", "آشتیان"),
    "دلیجانی": ("مرکزی", "دلیجان"),
    "خندابی": ("مرکزی", "خنداب"),
    "فراهانی": ("مرکزی", "فراهان"),
    "مامونیه‌ای": ("مرکزی", "مامونیه"),
}

# واحدهای اندازه گیری
UNIT_WORDS = [
    "پیمانه", "عدد", "قاشق غذاخوری", "قاشق چای‌خوری", "قاشق سوپخوری",
    "قاشق مرباخوری", "کاسه", "بشقاب", "لیوان", "فنجان",
    "گرم", "کیلو", "حبه", "ق غ","ق.غ", "ق.چ","ق چ", "ق م", "پ","ق غذاخوری",
    "ق چای خوری","قاشق غ","قاشق چای خوری","پنس","استکان","قاشق غذا خوری"
]

# الگوهای مقدار دلخواه
DESC_PATTERNS = [
    "به مقدار لازم", "به میزان لازم", "به میزان کافی",
    "به اندازه لازم", "مقداری", "به مقدار دلخواه", "کمی","بمقدار لازم",
    "به‌ اندازه نياز","به میزان دلخواه","مقدار لازم"
]

def detect_meal_type(title, ingredients_text):
    title_lower = title.lower()
    ingredients_lower = ingredients_text.lower() if ingredients_text else ""
    
    if any(w in title_lower for w in ["آبگوشت", "جگر", "خورشت", "خورش", "کباب", "کوفته", "دلمه", "کتلت", "شامی"]):
        return ["غذای اصلی گوشتی"]
    
    if any(w in title_lower for w in ["پلو", "برنج", "برنجی", "چلو"]):
        return ["غذای اصلی برنجی"]
    
    if any(w in title_lower for w in ["آش", "سوپ", "حلیم", "اشکنه"]):
        return ["آش و سوپ"]
    
    if any(w in title_lower for w in ["خوراک", "بورانی", "کوکو", "املت"]):
        if "گوشت" in ingredients_lower:
            return ["غذای اصلی گوشتی"]
        return ["غذای اصلی گیاهی"]
    
    if any(w in title_lower for w in ["شیرینی", "حلوا", "کیک", "بستنی", "دسر", "شله", "مهلبیه"]):
        return ["دسر"]
    
    if any(w in title_lower for w in ["نان", "ساجی", "لواش", "تافتون"]):
        return ["نان و شیرینی"]
    
    if any(w in title_lower for w in ["سالاد", "ترشی", "ماست"]):
        return ["پیش غذا و مکمل"]
    
    if "گوشت" in ingredients_lower:
        return ["غذای اصلی گوشتی"]
    
    if "برنج" in ingredients_lower:
        return ["غذای اصلی برنجی"]
    
    if "شکر" in ingredients_lower or "عسل" in ingredients_lower:
        return ["دسر"]
    
    return ["-"]

def detect_occasion(title, meal_type, ingredients=None, tags=None, season=None, time_of_day=None, is_weekend=False):
    title_lower = title.lower()
    ingredients_lower = ingredients.lower() if ingredients else ""
    
    MAIN_OCCASIONS = {
        "صبحانه": ["صبحانه", "ناشتا", "کله", "پاچه", "حلیم", "املت", "نیمرو", "پنیر"],
        "ناهار": ["ناهار", "چاشت", "ظهر", "پلو", "برنج", "چلو", "خورشت"],
        "شام": ["شام", "شبان", "کباب", "آبگوشت", "دیزی"],
        "عصرانه": ["عصرانه", "چای", "کیک", "شیرینی", "دسر"],
        "سحری": ["سحری", "رمضان"],
        "افطاری": ["افطاری", "افطار"]
    }

    primary_occasion = None

    if any(kw in title_lower for kw in MAIN_OCCASIONS["سحری"]):
        return "سحری"
    if any(kw in title_lower for kw in MAIN_OCCASIONS["افطاری"]):
        return "افطاری"

    for occasion, keywords in MAIN_OCCASIONS.items():
        if any(kw in title_lower for kw in keywords):
            primary_occasion = occasion
            break

    if not primary_occasion:
        if time_of_day == "صبح":
            return "صبحانه"
        if time_of_day == "عصر":
            return "عصرانه"
        return "شام" if not is_weekend else "ناهар"

    return primary_occasion

def parse_ingredient_line(line: str) -> dict:
    # First check for descriptive patterns
    for desc in DESC_PATTERNS:
        if desc in line:
            name = re.sub(r'[.:()۰-۹\d]+', '', line.replace(desc, '')).strip(' :-–،')
            return {"name": name, "amount": desc, "unit": None}

    # Convert Persian numbers to English
    line = line.translate(str.maketrans('۰۱۲۳۴۵۶۷۸۹', '0123456789'))
    
    # Remove extra spaces
    line = ' '.join(line.split())
    
    # Check for fractions and special characters first
    fraction_pattern = r'([½¾¼⅓⅔]|\d+/\d+)'
    fraction_match = re.search(fraction_pattern, line)
    
    if fraction_match:
        fraction = fraction_match.group(1)
        amount = WORD_TO_NUMBER.get(fraction, fraction)
        rest = line.replace(fraction, '').strip()
        
        # Find measurement unit
        unit = None
        for u in UNIT_WORDS:
            if u in rest:
                unit = u
                name = rest.replace(u, '').strip()
                break
        
        if not unit:
            name = rest.strip()
        
        return {"name": name, "amount": amount, "unit": unit}
    
    # Regular number parsing
    parts = re.split(r'(\d+\.?\d*)', line, maxsplit=1)
    if len(parts) > 1:
        amount = parts[1]
        rest = parts[2] if len(parts) > 2 else ''
        
        # Find measurement unit
        unit = None
        for u in UNIT_WORDS:
            if u in rest:
                unit = u
                name = rest.replace(u, '').strip()
                break
        
        if not unit:
            name = rest.strip()
        
        try:
            amount = float(amount) if '.' in amount else int(amount)
        except ValueError:
            amount = None
        
        return {"name": name, "amount": amount, "unit": unit}
    
    # Check for word numbers
    for word, num in WORD_TO_NUMBER.items():
        if word in line:
            amount = num
            rest = line.replace(word, '').strip()
            
            # Find measurement unit
            unit = None
            for u in UNIT_WORDS:
                if u in rest:
                    unit = u
                    name = rest.replace(u, '').strip()
                    break
            
            if not unit:
                name = rest.strip()
            
            return {"name": name, "amount": amount, "unit": unit}
    
    return {"name": line.strip(), "amount": None, "unit": None}

def extract_city_from_url(path: str, title: str = "") -> list:
    # Extract suffixes from URL
    url_suffixes = re.findall(r'-([\u0600-\u06FF()]+)', path)
    candidates = [s.strip('()') for s in url_suffixes]
    
    # Extract from title
    candidates += re.findall(r'\((.*?)\)', title)
    parts = title.strip().split()
    if parts:
        candidates.append(parts[-1])
    
    # Remove duplicates
    seen = []
    for c in candidates:
        if c and c not in seen:
            seen.append(c)
    
    # Convert to geographical locations
    results = []
    for suffix in seen:
        if suffix in SUFFIX_LOCATION_MAP:
            prov, city = SUFFIX_LOCATION_MAP[suffix]
            city_data = CITY_LOCATION_MAP.get(prov, {}).get(city, {})
            results.append({
                'province': prov,
                'city': city,
                'latitude': city_data.get('latitude'),
                'longitude': city_data.get('longitude')
            })
    
    return results

def split_instructions(instructions: str) -> list:
    """Split instructions into separate sentences"""
    parts = re.split(r'[.!؟]\s*', instructions)
    return [s.strip() for s in parts if len(s.strip()) > 5]

def extract_chishi_recipe(url: str) -> dict:
    try:
        # Fetch page content
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, 'html.parser')
        
        # Extract title
        title = soup.find('h2', class_='entry-title').text.strip()
        
        # Extract city and province
        raw_path = unquote(urlparse(url).path)
        locs = extract_city_from_url(raw_path, title)
        
        # Extract ingredients from table
        ingredients = []
        table = soup.find('table')
        if table:
            for row in table.find_all('tr')[1:]:  # Skip header row
                cols = row.find_all('td')
                if len(cols) >= 2:
                    ingredient = cols[0].text.strip()
                    amount = cols[1].text.strip()
                    parsed = parse_ingredient_line(f"{amount} {ingredient}" if amount else ingredient)
                    ingredients.append(parsed)
        
        # Extract cooking instructions
        instructions = []
        post_content = soup.find('div', class_='post-content')
        if post_content:
            # Find all h3 tags and their following paragraphs
            for h3 in post_content.find_all('h3'):
                if 'مرحله' in h3.text:
                    step_text = h3.text.strip()
                    next_p = h3.find_next('p')
                    step_desc = next_p.text.strip() if next_p else ""
                    instructions.append(f"{step_text}: {step_desc}")
        
        # Extract main image
        image = soup.find('img', class_='wp-image')['src'] if soup.find('img', class_='wp-image') else ""
        
        # Detect meal type and occasion
        ingredients_text = "\n".join([f"{i.get('amount', '')} {i.get('unit', '')} {i['name']}" for i in ingredients])
        meal_type = detect_meal_type(title, ingredients_text)
        occasion = detect_occasion(title, meal_type)
        
        return {
            'title': title,
            'locations': locs,
            'ingredients': ingredients,
            'instructions': instructions,
            'meal_type': meal_type,
            'occasion': occasion,
            'images': {
                'main': image,
                'step1': None,
                'step2': None
            },
            'source_url': url
        }
    except Exception as e:
        print(f"Error processing {url}: {str(e)}")
        return None

def process_recipes(urls: list, output_file: str):
    recipes = []
    for url in urls:
        recipe = extract_chishi_recipe(url)
        if recipe:
            recipes.append(recipe)
    
    # Save to JSON file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(recipes, f, ensure_ascii=False, indent=2)
    
    print(f"Saved {len(recipes)} recipes to {output_file}")

def normalize_city(city, province):
    if not city:
        return list(CITY_LOCATION_MAP.get(province, {}).keys())[0] if CITY_LOCATION_MAP.get(province) else None
    
    city = city.strip()
    
    replacements = {
        "خرم آباد": "خرم‌آباد",
        "خرماباد": "خرم‌آباد",
        "اسلام آباد": "اسلام‌آباد غرب",
        "قصرشیرین": "قصر شیرین",
        "سرپل ذهاب": "سرپل‌ذهاب",
        "سرپلذهاب": "سرپل‌ذهاب",
        "باغ ملک": "باغ‌ملک",
    }
    
    city = replacements.get(city, city)
    
    if province in CITY_LOCATION_MAP and city in CITY_LOCATION_MAP[province]:
        return city
    
    if province in CITY_LOCATION_MAP and CITY_LOCATION_MAP[province]:
        return list(CITY_LOCATION_MAP[province].keys())[0]
    
    return city

def extract_province_and_city_from_url(url, province_group=None):
    try:
        path = urlparse(url).path
        last_part = unquote(path.split('/')[-2])
        cleaned = re.sub(r'[^آ-یa-zA-Z]', '', last_part)

        for suffix, (province, city) in SUFFIX_LOCATION_MAP.items():
            if suffix in cleaned:
                return province, city

        for province, cities in CITY_LOCATION_MAP.items():
            if province in cleaned:
                for city in cities:
                    if city in cleaned:
                        return province, city
                return province, list(cities.keys())[0]

        if province_group and province_group in CITY_LOCATION_MAP:
            return province_group, list(CITY_LOCATION_MAP[province_group].keys())[0]

    except Exception as e:
        print(f"خطا در استخراج استان و شهر: {e}")

    return "نامشخص", "نامشخص"

def parse_ingredients(raw_html):
    ingredients = []
    for content in raw_html.select(".recipe-materials .mat-content"):
        name_tag = content.find("div", class_="fw-bold")
        amount_tag = content.find("div", class_="small")
        name = name_tag.text.strip() if name_tag else ""
        amount_unit = amount_tag.text.strip() if amount_tag else ""
        
        pattern = r'([\d٫\.\/\d]+)?\s*(ق\.?غ|ق\.?م|ق\.?چ|لیوان|فنجان|عدد|گرم|بسته|حبه|ورق|پیمانه|کاسه|بشقاب|قاشق غذاخوری|قاشق چای‌خوری|قاشق سوپخوری|قاشق مرباخوری)?'
        match = re.match(pattern, amount_unit)
        
        if match:
            amount = match.group(1).replace("٫", ".") if match.group(1) else None
            try:
                amount = float(amount) if amount else None
            except:
                pass
            unit = match.group(2) or None
        else:
            amount, unit = None, None
            
        ingredients.append({"name": name, "amount": amount, "unit": unit})
    return ingredients

def split_instructions(soup):
    instructions = []
    for step in soup.select(".recipe .text-justify"):
        for p in step.find_all(["p", "li"]):
            txt = p.get_text(strip=True)
            if txt and len(txt) > 5:
                instructions.append(txt)
    return instructions

def detect_meal_type(title, ingredients_text):
    text = f"{title} {ingredients_text}".lower()
    meal_types = {
        "غذای اصلی گوشتی": ["آبگوشت", "جگر", "کباب", "خورش", "کوفته", "دلمه"],
        "غذای اصلی برنجی": ["پلو", "برنج", "چلو"],
        "آش و سوپ": ["آش", "سوپ", "حلیم"],
        "دسر": ["شیرینی", "حلوا", "کیک", "دسر"],
        "نان و شیرینی": ["نان", "ساجی", "لواش"],
        "پیش غذا": ["سالاد", "ترشی", "ماست"]
    }
    
    for meal_type, keywords in meal_types.items():
        if any(keyword in text for keyword in keywords):
            return [meal_type]
    
    return ["غذای اصلی"]

def extract_recipe_from_canbo(url, province_group=None):
    try:
        res = requests.get(url, timeout=10)
        soup = BeautifulSoup(res.content, "html.parser")
        
        province, city = extract_province_and_city_from_url(url, province_group)
        norm_city = normalize_city(city, province)
        coord = CITY_LOCATION_MAP.get(province, {}).get(norm_city, {})

        title_tag = soup.find("h1")
        title = title_tag.get_text(strip=True).replace("دستور تهیه", "") if title_tag else "نامشخص"

        image_div = soup.find("div", class_="recipe image-box")
        style = image_div.get("style", "") if image_div else ""
        image_url = re.search(r"url\('(.+?)'\)", style)
        final_image = image_url.group(1) if image_url else ""

        ingredients = parse_ingredients(soup)
        instructions = split_instructions(soup)
        meal_type = detect_meal_type(title, " ".join([i["name"] for i in ingredients]))
        occasion = ["ناهار"]  

        return {
            "title": title,
            "locations": [{
                "province": province,
                "city": norm_city,
                "latitude": coord.get("latitude"),
                "longitude": coord.get("longitude")
            }],
            "ingredients": ingredients,
            "instructions": instructions,
            "meal_type": meal_type,
            "occasion": occasion,
            "images": {
                "main": final_image,
                "steps": []
            }
        }
    except Exception as e:
        print(f"خطا در پردازش URL {url}: {e}")
        return None

if __name__ == "__main__":
    urls = {
        "ایلام": [
            "https://mag.canbo.ir/recipe/ashe-masoa-2/",
            "https://mag.canbo.ir/recipe/ashe-peroshke/",
            "https://mag.canbo.ir/recipe/kofteh-sirabi/",
            "https://mag.canbo.ir/recipe/%D9%85%DA%A9%D8%B4-%D8%A7%DB%8C%D9%84%D8%A7%D9%85%DB%8C/",
            "https://mag.canbo.ir/recipe/%DA%A9%D8%A8%D9%87-%D8%A7%DB%8C%D9%84%D8%A7%D9%85%DB%8C/",
            "https://mag.canbo.ir/recipe/ashe-peroshke/",
            "https://mag.canbo.ir/recipe/%d8%a8%d8%a7%d8%a8%d8%a7%db%8c%d8%a7%d8%b1%db%8c/",
            "https://mag.canbo.ir/recipe/ash-shorba/",
        ]
    }

    for province, province_urls in urls.items():
        recipes = []
        for url in province_urls:
            recipe = extract_recipe_from_canbo(url, province)
            if recipe:
                recipes.append(recipe)
        
        if recipes:
            filename = f"canbo_{province}.json"
            with open(filename, "w", encoding="utf-8") as f:
                json.dump(recipes, f, ensure_ascii=False, indent=2)
            print(f"{province} Done :)")

خطا در پردازش URL https://mag.canbo.ir/recipe/%d8%a8%d8%a7%d8%a8%d8%a7%db%8c%d8%a7%d8%b1%db%8c/: HTTPSConnectionPool(host='mag.canbo.ir', port=443): Read timed out. (read timeout=10)
ایلام Done :)
